In [ ]:
# -*- coding: utf-8 -*-
# UI integrada: Aerogeofísica + SOM + Satélite (BDC/INPE/STAC)
# - Corrigido rasterio.sample (sem 'resampling')
# - Baixar itens STAC que intersectam a quadrícula -> sat_store[item_id]
# - Amostrar TCI/bandas (item único ou todos) para a grade interpolada

# ==== imports do seu projeto ====
from src import *                               # Build_mc, Upload_geof, pop_nodata, sintetic_grid, import_malha_cartog
from verde_source import regular, interp_at     # opcional (interp_at utilizado se presente)

# ==== libs ====
import os, re, json, types, importlib, warnings, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from matplotlib.colors import ListedColormap, BoundaryNorm
from shapely.geometry import Point, Polygon
from shapely.ops import transform as shp_transform

from tqdm import tqdm
import ipywidgets as W
from IPython.display import display, clear_output

from sklearn_som.som import SOM
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from pystac_client import Client
import rasterio
from rasterio.enums import Resampling
from pyproj import Transformer

warnings.filterwarnings("ignore")
%matplotlib inline

# ====================== ESTADO GLOBAL ======================
ESCALAS = ['25k','50k','100k','250k','1kk']

quadricula = {}         # {fid: { 'folha': Series(EPSG=...), 'gama_*': df, 'mag_*': df, 'geof_*': df, ... }}
data_grid = None        # nome da camada interpolada SOM (ex.: 'geof_1105_linear')
som_store = {}          # {k: {'som','imp','sca','feats','layer'}}
som_last_pred = None
bdc_items = []          # lista de pystac.Item da busca corrente
sat_store = {}          # { item_id: {'collection','datetime','bbox','assets':{name:path}, 'hrefs':{name:href}} }

# ====================== HELPERS GERAIS ======================
def _ids_from_mc(escala, filtro_regex=None):
    mc = import_malha_cartog(escala=escala)
    ids = mc['id_folha'].astype(str).tolist()
    if filtro_regex:
        pat = re.compile(filtro_regex, re.IGNORECASE)
        ids = [i for i in ids if pat.search(i)]
    return sorted(ids)

def _scan_layers_from_quadricula(q):
    layers = set()
    for _, blob in (q or {}).items():
        for k, v in blob.items():
            if isinstance(v, pd.DataFrame):
                layers.add(k)
    return tuple(sorted(layers))

def _available_columns(q, layers):
    cols = set()
    for _, blob in (q or {}).items():
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                cols.update([c for c in df.columns if c not in ('X','Y','E_utm','N_utm')])
    cols = sorted(cols, key=lambda c: (c!='MDT', c))
    return tuple(cols)

def _global_min_max_numeric(q, ids, layers, column, remove_neg=False):
    vals = []
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                s = df[column]
                if pd.api.types.is_numeric_dtype(s):
                    if remove_neg: s = s[s >= 0]
                    if s.size: vals.append(s.to_numpy())
    if not vals: return None, None
    v = np.concatenate(vals)
    if v.size == 0 or np.all(np.isnan(v)): return None, None
    return float(np.nanmin(v)), float(np.nanmax(v))

def _plot_layers_for_column(q, ids, layers, column, remove_neg=False):
    plt.figure(figsize=(12,9)); ax = plt.gca()
    is_num = False
    for fid in ids:
        for lay in layers:
            df = q.get(fid, {}).get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                is_num = pd.api.types.is_numeric_dtype(df[column]); break
        if is_num: break
    if is_num:
        vmin, vmax = _global_min_max_numeric(q, ids, layers, column, remove_neg)
        if vmin is not None and vmax is not None and vmin == vmax: vmin, vmax = vmin-1e-9, vmax+1e-9
        norm = colors.Normalize(vmin=vmin, vmax=vmax) if vmin is not None else None
        cmap = cm.get_cmap('terrain')
    for fid in ids:
        for lay in layers:
            df = q.get(fid, {}).get(lay)
            if not isinstance(df, pd.DataFrame) or column not in df.columns: continue
            d = df if not (is_num and remove_neg) else df[df[column] >= 0]
            if d.empty: continue
            if is_num:
                ax.scatter(d.X.values, d.Y.values, c=d[column].values, s=0.1, cmap=cmap, norm=norm, marker='H')
            else:
                codes, _ = pd.factorize(d[column], sort=True)
                ax.scatter(d.X.values, d.Y.values, c=codes, s=0.1, cmap='tab20', marker='H')
    ax.set_aspect('equal'); ax.set_title(f'Pré-visualização • {column} • {len(ids)} folha(s) • {", ".join(layers)}')
    if is_num and norm is not None:
        cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
        vmin, vmax = norm.vmin, norm.vmax; mid = (vmin+vmax)/2
        cbar.set_ticks([vmin, mid, vmax]); cbar.ax.set_yticklabels([f'{vmin:.3g}', f'{mid:.3g}', f'{vmax:.3g}'])
        cbar.set_label(f'{column} (min→máx)')
    plt.show()

def _make_discrete_cmap(n):
    base = plt.get_cmap('tab20')
    if hasattr(base, 'colors') and len(base.colors) >= n: return ListedColormap(base.colors[:n], name=f'tab20_{n}')
    return plt.get_cmap('nipy_spectral', n)

def _infer_suffix_from_names(*names):
    for nm in names or []:
        m = re.search(r'(\d{4})', str(nm) if nm else '')
        if m: return m.group(1)
    return '0000'

def _norm_name(s): return re.sub(r'[^a-z0-9]+','',str(s).lower())

_SYNONYMS = {
    'GMT': {'gmt','magigrf','magr','igrf','mag','gmtigrf'},
    'MDT': {'mdt','alte','altura'},
    'CTCOR': {'ctcor','ctc','ct'},
    'eTh': {'eth','eth_ppm','thc','th_ppm','ethppm','th'},
    'eU': {'eu','uc','u','euppm','u_ppm'},
    'KPERC': {'kperc','kc','k','kpct','k_percent'},
    'UTHRAZAO': {'uthrazao','uratio','u_th','u/th','u_th_ratio'},
    'UKRAZAO': {'ukrazao','u_k','u/k','u_k_ratio'},
    'THKRAZAO': {'thkrazao','th_k','th/k','th_k_ratio'},
}

def _find_source_column(df, canonical):
    want = _norm_name(canonical)
    for c in df.columns:
        if _norm_name(c) == want: return c
    for s in _SYNONYMS.get(canonical, set()):
        for c in df.columns:
            if _norm_name(c) == s: return c
    return None

def _source_order_for_feature(canonical):
    return ['mag','gama'] if canonical in ('GMT','MDT') else ['gama','mag']

# ============ INTERPOLAÇÃO para a grade ============
def _interpolate_current_selection(quad, ids, gama_key, mag_key, features, psize, algo, noneg=False):
    suf = _infer_suffix_from_names(gama_key, mag_key); out_name = f"geof_{suf}_{algo}"
    for fid in ids:
        blob = quad.get(fid, {})
        gdf = blob.get(gama_key); mdf = blob.get(mag_key)
        if gdf is None and mdf is None: continue
        xu, yu = sintetic_grid(quad, fid, psize=int(psize))
        sources = {}
        if isinstance(gdf, pd.DataFrame):
            gsrc = gdf.copy()
            if noneg:
                for c in gsrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(gsrc[c]):
                        gsrc.loc[gsrc[c] < 0, c] = np.nan
            sources['gama'] = (np.asarray(gsrc['X']), np.asarray(gsrc['Y']), gsrc)
        if isinstance(mdf, pd.DataFrame):
            msrc = mdf.copy()
            if noneg:
                for c in msrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(msrc[c]):
                        msrc.loc[msrc[c] < 0, c] = np.nan
            sources['mag'] = (np.asarray(msrc['X']), np.asarray(msrc['Y']), msrc)
        if not sources: continue
        out = {'X': xu, 'Y': yu}
        for f in features:
            arr = None
            for src in _source_order_for_feature(f):
                if src not in sources: continue
                x, y, df = sources[src]
                col = _find_source_column(df, f)
                if col is None: continue
                arr = interp_at(x, y, df[col].to_numpy(), xu, yu, algorithm=algo, extrapolate=True)
                break
            if arr is None: arr = np.full_like(xu, np.nan, dtype='float32')
            out[f] = arr
        quad[fid][out_name] = pd.DataFrame(out)
    return out_name

# ============ helpers UI ============
def _rescan_from_quadricula():
    q = globals().get('quadricula', {})
    layers = _scan_layers_from_quadricula(q); w_layers.options = layers
    global data_grid
    pick = (data_grid,) if data_grid and data_grid in layers else (layers[:1] if layers else ())
    w_layers.value = pick if pick else ()
    cols = _available_columns(q, w_layers.value) or ('MDT',)
    w_cols.options = cols
    w_cols.value = tuple([c for c in ('MDT',) if c in cols]) or ((cols[0],) if cols else ())
    # SOM widgets dependentes
    numeric = []
    for _, blob in q.items():
        for lay in w_layers.value:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in cols:
                    if c in df.columns and pd.api.types.is_numeric_dtype(df[c]): numeric.append(c)
    opts = sorted(set(numeric), key=lambda c: (c!='MDT', c)) or ['MDT']
    w_feats.options = opts
    keep = [c for c in w_feats.value if c in opts] or (['MDT'] if 'MDT' in opts else opts[:min(5,len(opts))])
    w_feats.value = tuple(keep)
    test_opts = []
    if data_grid:
        for fid, blob in q.items():
            if data_grid in blob and isinstance(blob[data_grid], pd.DataFrame):
                test_opts.append(fid)
    w_test_ids.options = tuple(sorted(test_opts))
    w_test_ids.value = tuple(sorted(test_opts))[:min(4, len(test_opts))]
    ks = sorted(list(som_store.keys()))
    w_k_apply.options = ks
    if ks: w_k_apply.value = ks[0]

def _normalize_xy(df):
    if not {'E_utm','N_utm'}.issubset(df.columns):
        if {'X','Y'}.issubset(df.columns): df = df.rename(columns={'X':'E_utm','Y':'N_utm'}).copy()
        else: raise ValueError("Camada sem 'X','Y' ou 'E_utm','N_utm'.")
    df = df.sort_values(['N_utm','E_utm'], ascending=[False, True], ignore_index=True, kind='mergesort')
    xs1d = np.sort(df['E_utm'].unique()); ys1d = np.sort(df['N_utm'].unique())
    nx, ny = xs1d.size, ys1d.size
    xs_mesh, ys_mesh = np.meshgrid(xs1d, ys1d)
    return df, xs_mesh, ys_mesh, nx, ny

def _build_matrix_for_fids(quad, features, layer, fids=None):
    fids_all = sorted(quad.keys()) if fids is None else list(fids)
    all_blocks, slc, metas = [], {}, {}
    k = 0
    for fid in fids_all:
        blob = quad.get(fid, {})
        if layer not in blob: continue
        df = blob[layer].copy()
        try:
            df, xs_mesh, ys_mesh, nx, ny = _normalize_xy(df)
        except Exception: continue
        metas[fid] = {'nx': nx, 'ny': ny, 'xs': xs_mesh, 'ys': ys_mesh}
        X = df[features].to_numpy(dtype='float32')
        if X.size == 0: continue
        all_blocks.append(X)
        slc[fid] = slice(k, k+len(X)); k += len(X)
    if not all_blocks: raise RuntimeError(f"Nenhuma folha com '{layer}' e as features escolhidas.")
    return np.vstack(all_blocks), slc, metas

def _qe(som, X_std):
    D = som.transform(X_std); return float(np.mean(np.min(D, axis=1)))

def _te_1d(som, X_std):
    D = som.transform(X_std)
    bmu = np.argmin(D, axis=1); D2 = D.copy(); D2[np.arange(D.shape[0]), bmu] = np.inf
    sbmu = np.argmin(D2, axis=1); return float(np.mean(np.abs(bmu - sbmu) > 1))

def _plot_classes(classes_by_fid, metas, n_clusters, flip_ns=False, titulo='Mapa preditivo (SOM)'):
    cmap = _make_discrete_cmap(n_clusters)
    bounds = np.arange(-0.5, n_clusters + 0.5, 1); norm = BoundaryNorm(bounds, ncolors=n_clusters, clip=True)
    fig, ax = plt.subplots(figsize=(10,10), facecolor='w')
    for fid in sorted(classes_by_fid.keys()):
        Z = classes_by_fid[fid];  Z = np.flipud(Z) if flip_ns else Z
        xs = metas[fid]['xs']; ys = metas[fid]['ys']
        ax.pcolormesh(xs, ys, Z, cmap=cmap, norm=norm, shading='nearest', rasterized=True)
    ax.set_aspect('equal'); ax.set_title(titulo)
    cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, ticks=np.arange(n_clusters), pad=0.01)
    cbar.ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)]); cbar.set_label('Classes')
    plt.tight_layout(); plt.show()

def _predict_per_folha(som, X_std, slc, metas):
    out = {}
    for fid, s in slc.items():
        y = som.predict(X_std[s]); ny, nx = metas[fid]['ny'], metas[fid]['nx']
        out[fid] = y.reshape(ny, nx)
    return out

def som_build_long_table(quad, layer, classes_by_fid, metas, atributos, fids=None):
    rows = []; fids_iter = list(classes_by_fid.keys()) if fids is None else list(fids)
    for fid in fids_iter:
        if fid not in classes_by_fid: continue
        Z = classes_by_fid[fid]; blob = quad.get(fid, {})
        if layer not in blob: continue
        df = blob[layer].copy(); df, xs, ys, nx, ny = _normalize_xy(df)
        Zv = Z.ravel(order='C') if Z.shape==(ny,nx) else np.ravel(Z)[:ny*nx]
        cols_keep = [a for a in atributos if a in df.columns]
        sub = pd.DataFrame({'fid': fid, 'E_utm': df['E_utm'].to_numpy(), 'N_utm': df['N_utm'].to_numpy(), 'classe': Zv.astype(int)})
        for a in cols_keep: sub[a] = df[a].to_numpy()
        rows.append(sub)
    if not rows: raise RuntimeError("Sem dados para tabela longa.")
    return pd.concat(rows, axis=0, ignore_index=True)

def plot_boxplots_por_atributo(df_long, atributos, classes=None, ncols=2, showfliers=False, rotation=45, sharey=True, figsize_cell=(4.0,3.2), suptitle=None):
    import math
    if classes is None: classes = sorted(pd.Series(df_long['classe']).dropna().unique())
    n = len(classes); ncols = max(1,int(ncols)); nrows = math.ceil(n/ncols)
    fig_w = max(6.0, figsize_cell[0]*ncols); fig_h = max(3.2, figsize_cell[1]*nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharey=sharey); axes = np.atleast_1d(axes).ravel()
    y_min = y_max = None
    if sharey:
        gvals=[]
        for c in classes:
            sub = df_long[df_long['classe']==c]
            for a in atributos:
                if a in sub.columns:
                    s = pd.to_numeric(sub[a], errors='coerce').dropna().values
                    if s.size: gvals.append(s)
        if gvals:
            gcat = np.concatenate(gvals); y_min, y_max = np.nanmin(gcat), np.nanmax(gcat)
    for i,c in enumerate(classes):
        ax = axes[i]; sub = df_long[df_long['classe']==c]
        vals, labels = [], []
        for a in atributos:
            if a in sub.columns:
                s = pd.to_numeric(sub[a], errors='coerce').dropna()
                if s.size: vals.append(s.values); labels.append(a)
        lab = int(c)+1 if isinstance(c,(int,np.integer)) else c
        if not vals: ax.set_title(f"Classe {lab} (sem dados)"); ax.axis("off")
        else:
            ax.boxplot(vals, labels=labels, showfliers=showfliers); ax.set_title(f"Classe {lab}")
            ax.set_xlabel("Atributo"); ax.set_ylabel("Valor"); ax.tick_params(axis='x', labelrotation=rotation)
            if y_min is not None and y_max is not None:
                pad = 0.03*(y_max-y_min if y_max!=y_min else 1.0); ax.set_ylim(y_min-pad, y_max+pad)
    for j in range(i+1, len(axes)): axes[j].axis("off")
    if suptitle: fig.suptitle(suptitle)
    plt.tight_layout(); plt.show(); return fig

# ============================ WIDGETS BASE ============================
w_escala = W.Dropdown(options=ESCALAS, value='100k', description='Escala')
w_filtro = W.Text(placeholder='ex.: SF23_YA', description='Filtro')
w_ids    = W.SelectMultiple(options=(), rows=10, description='Folhas')
w_selall = W.ToggleButton(value=False, description='Selecionar tudo', icon='check')
w_clear  = W.Button(description='Limpar', icon='trash')

w_ext    = W.IntSlider(min=0, max=2000, step=100, value=600, description='extend_size')
w_gama   = W.Dropdown(options=['gama_line_1105','gama_line_1089','gama_1039','gama_3022'], value='gama_line_1105', description='Gama')
w_mag    = W.Dropdown(options=['mag_line_1105','mag_line_1089','mag_1039','mag_3022'], value='mag_line_1105', description='Mag')
w_load   = W.Button(description='Carregar brutos', button_style='success', icon='download')

w_feats_interp = W.SelectMultiple(
    options=['GMT','CTCOR','eTh','eU','KPERC','UTHRAZAO','UKRAZAO','THKRAZAO','MDT'],
    value=('GMT','CTCOR','eTh','eU','KPERC','MDT'),
    rows=8, description='Features (grid)'
)
w_psize  = W.IntSlider(min=50, max=1000, step=50, value=100, description='Pixel (m)')
w_algo   = W.Dropdown(options=[('Linear','linear'),('Cúbico','cubic')], value='linear', description='Algoritmo')
w_nonegI = W.Checkbox(value=False, description='Negativos→NaN (grid)')
w_interpolar = W.Button(description='Interpolar grade', icon='shuffle')

w_layers = W.SelectMultiple(options=(), rows=6, description='Camadas')
w_cols   = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=6, description='Colunas')
w_nonegP = W.Checkbox(value=False, description='Remover negativos (preview)')
w_refresh = W.Button(description='Atualizar', icon='refresh')
w_plot   = W.Button(description='Pré-visualizar', icon='eye')

w_feats  = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=8, description='Features (SOM)')
w_sigma  = W.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.5, description='sigma')
w_iter   = W.IntSlider(min=500, max=30000, step=500, value=10000, description='max_iter')
w_seed   = W.IntSlider(min=0, max=9999, step=1, value=42, description='seed')
w_flip   = W.Checkbox(value=False, description='flip N-S no plot')
w_ks_train = W.SelectMultiple(options=tuple(range(3,31)), value=(8,12,16), rows=8, description='k p/ treinar')
w_train  = W.Button(description='Treinar SOM(s)', button_style='primary', icon='play')

w_test_ids = W.SelectMultiple(options=(), rows=8, description='Folhas (teste)')
w_seltest  = W.ToggleButton(value=False, description='Selecionar todas (teste)', icon='check')
w_k_apply  = W.Dropdown(options=[], description='k (aplicar)')
w_apply    = W.Button(description='Aplicar/Testar', icon='check-circle')
w_evalall  = W.Button(description='Comparar Ks (métricas)', icon='bar-chart')
w_clear_models = W.Button(description='Limpar modelos', icon='trash')

w_boxplots = W.Button(description='Boxplots por atributo', icon='bar-chart')
w_datagrid_label = W.HTML(value="<b>Camada SOM:</b> <i>—</i>")
w_models_label   = W.HTML(value="<b>Modelos treinados:</b> <i>—</i>")
w_out    = W.Output()

# ============================ CALLBACKS BASE ============================
def refresh_ids(*_):
    ids = _ids_from_mc(w_escala.value, w_filtro.value.strip() or None)
    w_ids.options = ids; w_selall.value = False

def on_selall_change(ch):
    if ch['name']=='value': w_ids.value = tuple(w_ids.options) if ch['new'] else ()

def on_seltest_change(ch):
    if ch['name']=='value': w_test_ids.value = tuple(w_test_ids.options) if ch['new'] else ()

def on_clear_clicked(_):
    w_filtro.value = ''; w_ids.value = ()

def on_load_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value: print('Selecione ao menos 1 folha.'); return
        print('# Montando grade…')
        quad = Build_mc(escala=w_escala.value, ID=list(w_ids.value), verbose=True)
        print('# Carregando dados brutos…')
        _g, _m = Upload_geof(quad, gama_xyz=w_gama.value, mag_xyz=w_mag.value, extend_size=int(w_ext.value))
        quad = pop_nodata(quad)
        globals()['quadricula'] = quad
        print(f'Folhas ativas: {len(quad)}')
        globals()['data_grid'] = None
        w_datagrid_label.value = "<b>Camada SOM:</b> <i>— (interpole primeiro)</i>"
        som_store.clear(); globals()['som_last_pred']=None; w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        print('Pronto. Dados brutos anexados. Agora execute a INTERPOLAÇÃO.')

def on_refresh_clicked(_):
    with w_out:
        clear_output()
        if 'quadricula' not in globals(): print('Carregue dados primeiro.'); return
        _rescan_from_quadricula(); print('Atualizado.')

def on_plot_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value: print('Selecione ao menos 1 folha.'); return
        if not w_layers.value: print('Nenhuma camada selecionada.'); return
        q = globals().get('quadricula', {})
        for col in w_cols.value:
            _plot_layers_for_column(q, w_ids.value, w_layers.value, col, remove_neg=w_nonegP.value)

def on_interpolar_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value: print('Selecione ao menos 1 folha.'); return
        feats_grid = list(w_feats_interp.value)
        if not feats_grid: print('Selecione ao menos 1 feature (grid).'); return
        q = globals().get('quadricula', {})
        if not q: print('Carregue dados brutos primeiro.'); return
        print(f"# Interpolando (algo={w_algo.value}, pixel={int(w_psize.value)} m)…")
        out_layer = _interpolate_current_selection(q, w_ids.value, w_gama.value, w_mag.value, feats_grid, int(w_psize.value), w_algo.value, w_nonegI.value)
        globals()['quadricula'] = q; globals()['data_grid'] = out_layer
        w_datagrid_label.value = f"<b>Camada SOM:</b> <code>{out_layer}</code>"
        print(f"→ Camada criada: {out_layer}")
        som_store.clear(); globals()['som_last_pred']=None; w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        if out_layer in w_layers.options: w_layers.value = (out_layer,)

def on_train_clicked(_):
    with w_out:
        clear_output()
        feats = list(w_feats.value)
        if not feats: print("Selecione ao menos 1 feature (SOM)."); return
        if not globals().get('data_grid'): print("Interpole a grade primeiro."); return
        layer = globals()['data_grid']; q = globals().get('quadricula', {})
        print(f"[TREINO] Montando matriz global de '{layer}'…")
        X_all, _, _ = _build_matrix_for_fids(q, feats, layer, fids=None)
        imp = SimpleImputer(strategy='median'); X_imp = imp.fit_transform(X_all)
        sca = StandardScaler().fit(X_imp); X_std = sca.transform(X_imp)
        ks = sorted(set(int(k) for k in w_ks_train.value)); 
        if not ks: print("Escolha ao menos um k."); return
        np.random.seed(int(w_seed.value))
        for k in ks:
            print(f" - SOM(k={k}, sigma={float(w_sigma.value)}, it={int(w_iter.value)})")
            som = SOM(m=int(k), n=1, sigma=float(w_sigma.value), dim=len(feats), max_iter=int(w_iter.value)); som.fit(X_std)
            som_store[k] = {'som': som, 'imp': imp, 'sca': sca, 'feats': feats, 'layer': layer}
        w_models_label.value = f"<b>Modelos treinados:</b> {', '.join(map(str, sorted(som_store.keys())))}"
        w_k_apply.options = sorted(list(som_store.keys()))
        if w_k_apply.options: w_k_apply.value = w_k_apply.options[0]
        print("Modelos treinados.")

def on_apply_clicked(_):
    with w_out:
        clear_output()
        if not som_store: print("Treine um SOM antes."); return
        if not w_test_ids.value: print("Selecione folhas para teste."); return
        k = int(w_k_apply.value); model = som_store.get(k)
        if model is None: print(f"k={k} não encontrado."); return
        feats = model['feats']; layer = model['layer']; q = globals().get('quadricula', {})
        print(f"[TESTE] Subset {len(w_test_ids.value)} folhas / layer '{layer}'…")
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e: print(str(e)); return
        X_te_std = model['sca'].transform(model['imp'].transform(X_te))
        qe = _qe(model['som'], X_te_std); te = _te_1d(model['som'], X_te_std)
        print(pd.DataFrame([{'k':k,'QE_test':qe,'TE_test':te}]).to_string(index=False))
        classes = _predict_per_folha(model['som'], X_te_std, slc_te, metas_te)
        _plot_classes(classes, metas_te, n_clusters=k, flip_ns=bool(w_flip.value),
                      titulo=f"SOM (aplicar) k={k} | sigma={float(w_sigma.value)} | it={int(w_iter.value)} | {layer}")
        globals()['som_last_pred'] = {'k':k, 'classes':classes, 'metas':metas_te, 'fids':tuple(w_test_ids.value), 'feats':tuple(feats), 'layer':layer}
        print("Predição salva: som_last_pred.")

def on_evalall_clicked(_):
    with w_out:
        clear_output()
        if not som_store or not w_test_ids.value: print("Treine/aplique SOM e selecione folhas."); return
        any_k = next(iter(som_store)); feats = som_store[any_k]['feats']; layer = som_store[any_k]['layer']
        q = globals().get('quadricula', {})
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e: print(str(e)); return
        rows=[]
        for k, model in sorted(som_store.items()):
            if model['feats']!=feats or model['layer']!=layer: rows.append({'k':k,'QE_test':np.nan,'TE_test':np.nan,'obs':'incompatível'}); continue
            X_te_std = model['sca'].transform(model['imp'].transform(X_te))
            rows.append({'k':k,'QE_test':_qe(model['som'],X_te_std),'TE_test':_te_1d(model['som'],X_te_std)})
        print(pd.DataFrame(rows).sort_values('QE_test', ascending=True, na_position='last').to_string(index=False))

def on_clear_models_clicked(_):
    som_store.clear(); globals()['som_last_pred']=None
    w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"; w_k_apply.options=[]
    with w_out: clear_output(); print("Modelos apagados.")

def on_boxplots_clicked(_):
    with w_out:
        clear_output()
        if not som_store or globals().get('som_last_pred') is None:
            print("Treine e aplique um SOM antes."); return
        lp = globals()['som_last_pred']; k=lp['k']; classes=lp['classes']; metas=lp['metas']; fids=lp['fids']; feats=list(lp['feats']); layer=lp['layer']
        q = globals().get('quadricula', {})
        try:
            df_long = som_build_long_table(q, layer, classes, metas, atributos=feats, fids=fids)
        except RuntimeError as e: print(str(e)); return
        print(f"[Boxplots] {len(fids)} folha(s) | k={k} | layer='{layer}' | atributos={feats}")
        plot_boxplots_por_atributo(df_long, atributos=feats, ncols=2, showfliers=False)

# liga
w_escala.observe(refresh_ids, names='value')
w_filtro.observe(refresh_ids, names='value')
w_selall.observe(on_selall_change, names='value')
w_seltest.observe(on_seltest_change, names='value')
w_clear.on_click(on_clear_clicked)
w_load.on_click(on_load_clicked)
w_refresh.on_click(on_refresh_clicked)
w_plot.on_click(on_plot_clicked)
w_interpolar.on_click(on_interpolar_clicked)
w_train.on_click(on_train_clicked)
w_apply.on_click(on_apply_clicked)
w_evalall.on_click(on_evalall_clicked)
w_clear_models.on_click(on_clear_models_clicked)
w_boxplots.on_click(on_boxplots_clicked)
refresh_ids()

# ============================ BDC / STAC (INPE) ============================
BDC_ENDPOINT = "https://data.inpe.br/bdc/stac/v1"

def _aoi_bbox_from_ids(escala, ids):
    if not ids: return None
    gdf = import_malha_cartog(escala=escala)
    gdf = gdf[gdf['id_folha'].astype(str).isin([str(i) for i in ids])].copy()
    if gdf.empty: return None
    try: gdf = gdf.to_crs(4326)
    except Exception: pass
    minx,miny,maxx,maxy = gdf.total_bounds
    return [float(minx), float(miny), float(maxx), float(maxy)]

def _bdc_list_collections(pattern=None):
    cli = Client.open(BDC_ENDPOINT)
    cols = [c.id for c in cli.get_collections()]
    if pattern:
        pat = re.compile(pattern, re.IGNORECASE); cols = [c for c in cols if pat.search(c)]
    return sorted(cols)

def _bdc_search_items(collections, bbox, dt_range, cloud_min, cloud_max, limit, sort_dir):
    cli = Client.open(BDC_ENDPOINT)
    q = {"eo:cloud_cover": {"gte": int(cloud_min), "lte": int(cloud_max)}}
    sortby = ["properties.datetime"] if sort_dir == "asc" else ["-properties.datetime"]
    search = cli.search(collections=list(collections), bbox=bbox, datetime=dt_range, query=q, sortby=sortby, max_items=int(limit))
    return list(search.items())

def _bdc_pick_visual_asset(item):
    for key in ("tci","visual","overview","thumbnail"):
        a = item.assets.get(key)
        if a and a.href: return a.href, key
    for trip in (("B4","B3","B2"),("red","green","blue")):
        if all(k in item.assets for k in trip): return item.assets[trip[0]].href, trip[0]
    return None

def _bdc_preview_thumbs(items, max_show=12):
    n = min(len(items), max_show)
    if n == 0: print("Nenhum item."); return
    ncols = 4; nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 2.8*nrows)); axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        it = items[i]; ax = axes[i]; ax.axis("off")
        pair = _bdc_pick_visual_asset(it)
        title = f"{it.collection_id}\n{getattr(it,'datetime',None).date() if getattr(it,'datetime',None) else '—'}"
        ax.set_title(title, fontsize=9)
        if pair is None: ax.text(0.5,0.5,"sem preview",ha="center",va="center"); continue
        href, _ = pair
        try:
            r = requests.get(href, timeout=15); r.raise_for_status()
            from PIL import Image; from io import BytesIO
            img = Image.open(BytesIO(r.content)); ax.imshow(img)
        except Exception as e:
            ax.text(0.5,0.5,f"erro preview\n{e}",ha="center",va="center",fontsize=8)
    for j in range(i+1, len(axes)): axes[j].axis("off")
    plt.tight_layout(); plt.show()

# -------- Amostragem de bandas na grade --------
def _grid_epsg_from_blob(blob):
    v = blob.get('folha', None)
    if v is not None:
        for key in ('EPSG','epsg'):
            if hasattr(v, key): 
                try: return int(getattr(v, key))
                except Exception: pass
            if isinstance(v, dict) and key in v:
                try: return int(v[key])
                except Exception: pass
    if 'EPSG' in blob:
        try: return int(blob['EPSG'])
        except Exception: pass
    raise RuntimeError("Não foi possível inferir o EPSG da folha.")

def _open_remote_raster(href):
    try: return rasterio.open(href)
    except Exception: pass
    if not href.startswith('/vsicurl/'): return rasterio.open('/vsicurl/' + href)
    raise


def _resolve_band_assets(item, bands_text):
    """
    Converte a string de bandas em [(name, href, idxs)].
    Suporta:
      - 'tci'/'visual' (RGB, índices (1,2,3))
      - nomes exatos de assets do item (uma banda)
      - apelidos: red/green/blue -> B4/B3/B2 (ou B04/B03/B02)
      - padrões: B8, band8, B08 etc.
    """
    wanted = [b.strip() for b in str(bands_text).split(',') if b.strip()]
    out = []

    # mapa case-insensitive das chaves de asset
    assets_ci = {k.lower(): k for k in item.assets.keys()}

    def _pick(*keys):
        """Tenta retornar (asset_key_real, href) para a primeira key disponível."""
        for k in keys:
            kk = assets_ci.get(k.lower())
            if kk:
                href = getattr(item.assets[kk], "href", None)
                if href:
                    return kk, href
        return None, None

    for w in wanted:
        lw = w.lower()

        # 1) TCI / VISUAL (RGB)
        if lw == 'tci':
            k, href = _pick('tci', 'visual')
            if not href:
                has_rgb = (
                    (assets_ci.get('b4') and assets_ci.get('b3') and assets_ci.get('b2')) or
                    (assets_ci.get('b04') and assets_ci.get('b03') and assets_ci.get('b02'))
                )
                if has_rgb:
                    raise RuntimeError("Item sem 'tci'/'visual'. Selecione B4,B3,B2 (ou B04,B03,B02).")
                raise RuntimeError("Item não oferece 'tci'/'visual'.")
            out.append(('tci', href, (1, 2, 3)))
            continue

        # 2) nome exato do asset
        kk = assets_ci.get(lw)
        if kk:
            href = item.assets[kk].href
            out.append((kk, href, (1,)))
            continue

        # 3) apelidos RGB
        if lw in ('red', 'b4', 'b04', 'band4'):
            k, href = _pick('B4', 'B04', 'red')
            if href:
                out.append(('red', href, (1,)))
                continue

        if lw in ('green', 'b3', 'b03', 'band3'):
            k, href = _pick('B3', 'B03', 'green')
            if href:
                out.append(('green', href, (1,)))
                continue

        if lw in ('blue', 'b2', 'b02', 'band2'):
            k, href = _pick('B2', 'B02', 'blue')
            if href:
                out.append(('blue', href, (1,)))
                continue

        # 4) padrão genérico: B8, B08, band8, band08, etc.
        m = re.fullmatch(r'b(?:and)?0?(\d+)', lw)
        if m:
            n = int(m.group(1))
            k, href = _pick(f'B{n}', f'B{n:02d}')
            if href:
                out.append((f'B{n}', href, (1,)))
                continue

        # 5) não achou
        raise RuntimeError(f"Banda/asset '{w}' não encontrada.")

    return out

def _sample_asset_into_layer(quad, fids, layer_name, href, band_idxs=(1,), prefix='sat'):
    """
    Amostra 1+ bandas do 'href' sobre os pontos (X,Y) do DataFrame 'layer_name'.
    Usa sample() do rasterio (nearest). Colunas criadas: <prefix>, <prefix>_r/g/b, etc.
    """
    with _open_remote_raster(href) as ds:
        if ds.crs is None: raise RuntimeError("GeoTIFF sem CRS.")
        ok_cols = 0
        for fid in fids:
            blob = quad.get(fid, {})
            df = blob.get(layer_name)
            if not isinstance(df, pd.DataFrame) or not {'X','Y'}.issubset(df.columns): continue
            try: epsg_grid = _grid_epsg_from_blob(blob)
            except Exception as e: print(f" - {fid}: erro EPSG → {e}"); continue
            tr = Transformer.from_crs(f"EPSG:{epsg_grid}", ds.crs, always_xy=True)
            xx, yy = tr.transform(df['X'].to_numpy(), df['Y'].to_numpy())
            # amostragem em blocos
            def _batched(xa, ya, bs=200000):
                for i in range(0, xa.size, bs): yield xa[i:i+bs], ya[i:i+bs]
            for j, b in enumerate(band_idxs, 1):
                vals = np.full(df.shape[0], np.nan, dtype='float32'); k = 0
                try:
                    for xb, yb in _batched(xx, yy):
                        pts = list(zip(xb, yb))
                        # rasterio.sample não aceita 'resampling' -> nearest
                        it = ds.sample(pts, indexes=b)  # <--- FIX
                        out = np.fromiter((row[0] for row in it), dtype='float32', count=xb.size)
                        vals[k:k+xb.size] = out; k += xb.size
                except Exception as e:
                    print(f" - {fid}: erro amostrando banda {b} → {e}"); continue
                # nome de coluna
                if len(band_idxs)==3:
                    suffix = ('r','g','b')[j-1] if j<=3 else f'b{j}'
                    col = f"{prefix}_{suffix}"
                elif len(band_idxs)==1:
                    col = f"{prefix}"
                else:
                    col = f"{prefix}_b{b}"
                df[col] = vals.astype('float32', copy=False); ok_cols += 1
        return ok_cols

# -------- Download e cache local de assets STAC --------
def _download_assets_for_item(item, bands_text, outdir="satellite_bdc"):
    os.makedirs(outdir, exist_ok=True)
    bands = _resolve_band_assets(item, bands_text)
    assets_local = {}; hrefs = {}
    base_dir = os.path.join(outdir, f"{item.collection_id}_{item.id}")
    os.makedirs(base_dir, exist_ok=True)
    for name, href, _idxs in bands:
        fname = os.path.basename(href.split('?')[0])
        fpath = os.path.join(base_dir, f"{name}_{fname}")
        if not os.path.exists(fpath):
            with requests.get(href, stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(fpath, "wb") as f:
                    for ch in r.iter_content(1<<20):
                        if ch: f.write(ch)
        assets_local[name] = fpath; hrefs[name] = href
    # guarda no sat_store
    sat_store[item.id] = {
        'collection': item.collection_id,
        'datetime': getattr(item, 'datetime', None),
        'bbox': getattr(item, 'bbox', None) or getattr(item, 'properties', {}).get('bbox'),
        'assets': assets_local,
        'hrefs': hrefs
    }
    return assets_local

# -------- UI BDC --------
w_bdc_filter = W.Text(placeholder='regex (ex.: landsat|sentinel|cbers)', description='Filtro')
w_bdc_list   = W.Button(description='Listar coleções', icon='list')
w_bdc_cols   = W.SelectMultiple(options=(), rows=8, description='Coleções')

w_bdc_date   = W.Text(value='2018-01-01/2025-12-31', description='Data (UTC)')
w_bdc_cloud  = W.IntRangeSlider(value=[0,100], min=0, max=100, step=1, description='Nuvens (%)')
w_bdc_limit  = W.IntSlider(value=20, min=1, max=200, step=1, description='Limite')
w_bdc_sort   = W.Dropdown(options=[('Mais antigo','asc'),('Mais recente','desc')], value='desc', description='Ordenar')

w_bdc_search = W.Button(description='Buscar itens', icon='search', button_style='info')
w_bdc_prev   = W.Button(description='Thumbnails', icon='image')
w_bdc_save   = W.Button(description='Baixar VISUAL', icon='download')

w_bdc_item    = W.Dropdown(options=(), description='Item', disabled=True)
w_bdc_bands   = W.Text(value='tci', description='Bandas')      # 'tci' | 'B4,B3,B2' | 'B8' | 'red,green,blue'
w_bdc_prefix  = W.Text(value='sat', description='Prefixo')
w_bdc_sample  = W.Button(description='Amostrar item', icon='plus-square', button_style='warning')

# novos botões p/ baixar / amostrar em lote
w_bdc_dl_all  = W.Button(description='Baixar itens (todos)', icon='download', button_style='success')
w_bdc_sm_all  = W.Button(description='Amostrar itens (todos)', icon='plus-square')
w_bdc_out     = W.Output()

def _format_item_label(it, i):
    coll = getattr(it, "collection_id", "") or ""
    dt   = getattr(it, "datetime", None)
    dts  = (dt.date().isoformat() if hasattr(dt, "date") else str(dt)) if dt else "—"
    props = getattr(it, "properties", {}) or {}
    cc = props.get("eo:cloud_cover") or props.get("cloud_cover")
    cc_str = (f"{cc:.0f}%" if isinstance(cc,(int,float)) else "—")
    return f"{i:02d} | {coll} | {dts} | clouds {cc_str}"

def on_bdc_list_clicked(_):
    with w_bdc_out:
        clear_output()
        try:
            cols = _bdc_list_collections(w_bdc_filter.value.strip() or None)
            if not cols: print("Nenhuma coleção encontrada.")
            else:
                w_bdc_cols.options = tuple(cols)
                print(f"{len(cols)} coleção(ões). Selecione e pesquise.")
        except Exception as e:
            print("Erro ao listar coleções:", e)

def on_bdc_search_clicked(_):
    with w_bdc_out:
        clear_output()
        if not w_bdc_cols.value: print("Selecione coleções."); return
        bbox = _aoi_bbox_from_ids(w_escala.value, list(w_ids.value))
        if not bbox: print("Selecione folhas (à esquerda) para definir a AOI)."); return
        print("AOI (bbox WGS84):", bbox)
        try:
            items = _bdc_search_items(
                collections=w_bdc_cols.value, bbox=bbox, dt_range=w_bdc_date.value.strip(),
                cloud_min=w_bdc_cloud.value[0], cloud_max=w_bdc_cloud.value[1],
                limit=int(w_bdc_limit.value), sort_dir=w_bdc_sort.value
            )
        except Exception as e:
            print("Erro na busca STAC:", e); return
        globals()['bdc_items'] = items
        print(f"Encontrados {len(items)} item(ns). Use 'Thumbnails' ou selecione um Item.")
        labels = [_format_item_label(it, i) for i,it in enumerate(items)]
        w_bdc_item.options = list(zip(labels, range(len(items))))
        w_bdc_item.disabled = (len(items)==0)
        if items: w_bdc_item.value = 0

def on_bdc_prev_clicked(_):
    with w_bdc_out: clear_output(); _bdc_preview_thumbs(globals().get('bdc_items', []), max_show=16)

def on_bdc_save_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items: print("Faça a busca primeiro."); return
        os.makedirs("satellite_bdc", exist_ok=True)
        saved=[]
        for it in items:
            pair = _bdc_pick_visual_asset(it)
            if not pair: continue
            href, key = pair
            name = os.path.basename(href.split('?')[0])
            fpath = os.path.join("satellite_bdc", f"{it.collection_id}_{it.id}_{key}_{name}")
            try:
                if not os.path.exists(fpath):
                    with requests.get(href, stream=True, timeout=60) as r:
                        r.raise_for_status()
                        with open(fpath,"wb") as f:
                            for ch in r.iter_content(1<<20):
                                if ch: f.write(ch)
                saved.append(fpath)
            except Exception as e:
                print(f"[WARN] Falha ao baixar {href}: {e}")
        if saved:
            print("Arquivos salvos:"); [print(" -",p) for p in saved]
        else:
            print("Nenhum asset visual pôde ser baixado.")

def on_bdc_sample_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items: print("Busque itens primeiro."); return
        idx = int(w_bdc_item.value)
        if not (0 <= idx < len(items)): print(f"Índice inválido 0..{len(items)-1}."); return
        if not globals().get('data_grid'): print("Interpole a grade (crie a camada SOM)."); return
        layer = globals()['data_grid']; q = globals().get('quadricula', {})
        fids_target = [fid for fid,blob in q.items() if layer in blob]
        item = items[idx]
        try:
            bands = _resolve_band_assets(item, w_bdc_bands.value)
        except Exception as e:
            print("Bandas:", str(e)); return
        print(f"Amostrando {[b[0] for b in bands]} → '{layer}' em {len(fids_target)} folha(s)…")
        total_cols=0
        for name, href, idxs in bands:
            try:
                cols = _sample_asset_into_layer(q, fids_target, layer_name=layer, href=href, band_idxs=tuple(idxs), prefix=(w_bdc_prefix.value or name))
                total_cols += cols; print(f"  - OK {name}: {cols} coluna(s).")
            except Exception as e:
                print(f"  - {name}: erro → {e}")
        if total_cols==0:
            print("Nenhuma coluna criada (verifique EPSG/GeoTIFF).")
        else:
            globals()['quadricula']=q; _rescan_from_quadricula(); print("Pronto. Novas colunas disponíveis no SOM.")

def on_bdc_dl_all_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items: print("Busque itens primeiro."); return
        bands_text = w_bdc_bands.value
        print(f"Baixando {len(items)} item(ns) ({bands_text})…")
        ok=0
        for it in items:
            try:
                local = _download_assets_for_item(it, bands_text, outdir="satellite_bdc")
                print(f" - {it.id}: {list(local.keys())}")
                ok += 1
            except Exception as e:
                print(f" - {it.id}: erro → {e}")
        print(f"Concluído. {ok}/{len(items)} item(ns) armazenados em sat_store.")

def on_bdc_sm_all_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items: print("Busque itens primeiro."); return
        if not globals().get('data_grid'): print("Interpole a grade (crie a camada SOM)."); return
        layer = globals()['data_grid']; q = globals().get('quadricula', {})
        fids_target = [fid for fid,blob in q.items() if layer in blob]
        bands_text = w_bdc_bands.value
        print(f"Amostrar TODOS os itens ({len(items)}), bandas={bands_text} → layer '{layer}' …")
        total_cols = 0; it_done = 0
        for it in items:
            try:
                bands = _resolve_band_assets(it, bands_text)
                for name, href, idxs in bands:
                    cols = _sample_asset_into_layer(q, fids_target, layer_name=layer, href=href, band_idxs=tuple(idxs), prefix=(w_bdc_prefix.value or name))
                    total_cols += cols
                it_done += 1
            except Exception as e:
                print(f" - {it.id}: erro → {e}")
        if total_cols==0:
            print("Nenhuma coluna criada (verifique EPSG/GeoTIFF).")
        else:
            globals()['quadricula']=q; _rescan_from_quadricula()
            print(f"OK. {it_done}/{len(items)} itens amostrados; {total_cols} coluna(s) adicionada(s).")

# liga BDC
w_bdc_list.on_click(on_bdc_list_clicked)
w_bdc_search.on_click(on_bdc_search_clicked)
w_bdc_prev.on_click(on_bdc_prev_clicked)
w_bdc_save.on_click(on_bdc_save_clicked)
w_bdc_sample.on_click(on_bdc_sample_clicked)
w_bdc_dl_all.on_click(on_bdc_dl_all_clicked)
w_bdc_sm_all.on_click(on_bdc_sm_all_clicked)

# painel BDC
bdc_controls = W.VBox([
    W.HBox([w_bdc_filter, w_bdc_list]),
    W.HBox([w_bdc_cols]),
    W.HBox([w_bdc_date, w_bdc_cloud, w_bdc_limit, w_bdc_sort]),
    W.HBox([w_bdc_search, w_bdc_prev, w_bdc_save]),
    W.HBox([w_bdc_item, w_bdc_bands, w_bdc_prefix, w_bdc_sample]),
    W.HBox([w_bdc_dl_all, w_bdc_sm_all]),
    w_bdc_out
])

# ============================ LAYOUT FINAL ============================
left = W.VBox([
    W.HBox([w_escala, w_filtro]),
    W.HBox([w_ids, W.VBox([w_selall, w_clear, w_ext, w_gama, w_mag, w_load, w_refresh, w_plot])]),
    W.HTML("<hr><b>Interpolação para grade</b>"),
    W.HBox([w_feats_interp, W.VBox([w_psize, w_algo, w_nonegI, w_interpolar])]),
    w_datagrid_label,
    W.HTML("<hr><b>Imagens de Satélite — BDC/INPE (STAC)</b>"),
    bdc_controls,
])

mid = W.VBox([
    W.HTML("<b>Pré-visualização</b>"),
    W.HBox([w_layers, w_cols]),
    w_nonegP,
    W.HTML("<hr><b>SOM — Treino</b>"),
    w_feats,
    W.HBox([w_sigma, w_iter, w_seed]),
    W.HBox([w_ks_train, w_train]),
    w_models_label
])

right = W.VBox([
    W.HTML("<b>SOM — Teste/Aplicação</b>"),
    W.HBox([w_test_ids, W.VBox([w_seltest, w_k_apply, w_apply, w_evalall, w_flip, w_clear_models, w_boxplots])])
])

ui = W.VBox([W.HBox([left, mid, right]), w_out])
display(ui)
# === UI ÚNICA: Carregar → Interpolar grade → Pré-visualizar → SOM (treino/teste) ===
# Requisitos no ambiente: Build_mc, Upload_geof, pop_nodata, sintetic_grid, interp_at

import re
import numpy as np
import pandas as pd
import geopandas as gpd
import ipywidgets as W
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

from matplotlib import cm, colors
from matplotlib.colors import ListedColormap, BoundaryNorm

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn_som.som import SOM

# ---------------- util ----------------
ESCALAS = ['25k','50k','100k','250k','1kk']

def _ids_from_mc(escala, filtro_regex=None):
    mc = import_malha_cartog(escala=escala)
    ids = mc['id_folha'].astype(str).tolist()
    if filtro_regex:
        pat = re.compile(filtro_regex, re.IGNORECASE)
        ids = [i for i in ids if pat.search(i)]
    return sorted(ids)

def _scan_layers_from_quadricula(q):
    layers = set()
    for fid, blob in (q or {}).items():
        for k, v in blob.items():
            if isinstance(v, pd.DataFrame):
                layers.add(k)
    return tuple(sorted(layers))

def _available_columns(q, layers):
    cols = set()
    for fid, blob in (q or {}).items():
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in df.columns:
                    if c in ('X','Y','E_utm','N_utm'):
                        continue
                    cols.add(c)
    cols = sorted(cols, key=lambda c: (c!='MDT', c))
    return tuple(cols)

def _global_min_max_numeric(q, ids, layers, column, remove_negatives=False):
    vals = []
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                s = df[column]
                if pd.api.types.is_numeric_dtype(s):
                    if remove_negatives:
                        s = s[s >= 0]
                    if s.size:
                        vals.append(s.to_numpy())
    if not vals:
        return None, None
    v = np.concatenate(vals)
    if v.size == 0 or np.all(np.isnan(v)):
        return None, None
    return float(np.nanmin(v)), float(np.nanmax(v))

def _plot_layers_for_column(q, ids, layers, column, remove_negatives=False):
    plt.figure(figsize=(12, 9))
    ax = plt.gca()
    is_numeric = False
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                is_numeric = pd.api.types.is_numeric_dtype(df[column])
                break
        if is_numeric:
            break

    if is_numeric:
        vmin, vmax = _global_min_max_numeric(q, ids, layers, column, remove_negatives)
        if vmin is not None and vmax is not None and vmin == vmax:
            eps = 1e-9
            vmin, vmax = vmin - eps, vmax + eps
        norm = colors.Normalize(vmin=vmin, vmax=vmax) if vmin is not None else None
        cmap = cm.get_cmap('terrain')

    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if not isinstance(df, pd.DataFrame) or column not in df.columns:
                continue
            d = df
            if is_numeric and remove_negatives:
                d = d[d[column] >= 0]
                if d.empty:
                    continue
            if is_numeric:
                ax.scatter(d.X.values, d.Y.values, c=d[column].values,
                           s=0.1, cmap=cmap, norm=norm, marker='H')
            else:
                codes, _ = pd.factorize(d[column], sort=True)
                ax.scatter(d.X.values, d.Y.values, c=codes, s=0.1,
                           cmap='tab20', marker='H')

    ax.set_aspect('equal')
    ax.set_title(f'Pré-visualização • {column} • {len(ids)} folha(s) • camadas: {", ".join(layers)}')
    plt.axis('scaled')

    if is_numeric and norm is not None:
        cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
        vmin, vmax = norm.vmin, norm.vmax
        mid = (vmin + vmax) / 2.0
        cbar.set_ticks([vmin, mid, vmax])
        cbar.ax.set_yticklabels([f'{vmin:.3g}', f'{mid:.3g}', f'{vmax:.3g}'])
        cbar.set_label(f'{column} (min→máx)', rotation=90)
    plt.show()

def _make_discrete_cmap(n):
    base = plt.get_cmap('tab20')
    if hasattr(base, 'colors') and len(base.colors) >= n:
        return ListedColormap(base.colors[:n], name=f'tab20_{n}')
    return plt.get_cmap('nipy_spectral', n)

def _infer_suffix_from_names(*names):
    for nm in names:
        if not nm:
            continue
        m = re.search(r'(\d{4})', str(nm))
        if m:
            return m.group(1)
    return '0000'

# ---------- helpers de mapeamento de nomes ----------
def _norm_name(name: str) -> str:
    """normaliza nome de coluna (minúsculas, só [a-z0-9])."""
    import re
    return re.sub(r'[^a-z0-9]+', '', str(name).lower())

# sinônimos por feature canônica (chave = como você quer que apareça no df final)
_SYNONYMS = {
    'GMT'      : {'gmt','magigrf','magr','igrf','mag','gmtigrf','magig rf','magigrf'},  # robustez
    'MDT'      : {'mdt','alte','altura'},
    'CTCOR'    : {'ctcor','ctc','ct'},
    'eTh'      : {'eth','eth_ppm','thc','th_ppm','ethppm','th'},
    'eU'       : {'eu','uc','u','euppm','u_ppm'},
    'KPERC'    : {'kperc','kc','k','kpct','k_percent'},
    'UTHRAZAO' : {'uthrazao','uratio','u_th','u/th','u_th_ratio'},
    'UKRAZAO'  : {'ukrazao','u_k','u/k','u_k_ratio'},
    'THKRAZAO' : {'thkrazao','th_k','th/k','th_k_ratio'},
}

def _find_source_column(df: pd.DataFrame, canonical: str) -> str | None:
    """
    Procura em df uma coluna equivalente à feature canônica (por sinônimos).
    Retorna o nome real da coluna no df ou None.
    """
    want = _norm_name(canonical)
    for c in df.columns:  # match direto
        if _norm_name(c) == want:
            return c
    # por sinônimos
    syns = _SYNONYMS.get(canonical, set())
    cols_norm = { _norm_name(c): c for c in df.columns }
    for s in syns:
        if s in cols_norm:
            return cols_norm[s]
    return None

def _source_order_for_feature(canonical: str) -> list[str]:
    """
    Preferência de fonte para cada feature canônica.
    """
    if canonical in ('GMT', 'MDT'):
        return ['mag', 'gama']   # GMT tende a vir do magnético; MDT idem
    return ['gama', 'mag']       # demais: primeiro gama, depois magnético

# --------- INTERPOLAÇÃO p/ grade sintética ---------
def _interpolate_current_selection(quadricula, ids, gama_key, mag_key, features, psize, algo, noneg=False):
    """
    Para cada folha:
      - monta grade sintética (psize em metros)
      - interpola cada feature canônica a partir de gama/mag (com sinônimos)
      - salva em 'geof_<sufixo>_<algo>' com colunas: X, Y + features (sempre presentes)
    """
    suf = _infer_suffix_from_names(gama_key, mag_key)
    out_name = f"geof_{suf}_{algo}"

    for fid in ids:
        blob = quadricula.get(fid, {})
        gama_df = blob.get(gama_key)
        mag_df  = blob.get(mag_key)

        if (gama_df is None) and (mag_df is None):
            continue

        # grid sintética por folha
        xu, yu = sintetic_grid(quadricula, fid, psize=int(psize))

        # preparação de fontes
        sources = {}
        if isinstance(gama_df, pd.DataFrame):
            gsrc = gama_df.copy()
            if noneg:
                for c in gsrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(gsrc[c]):
                        gsrc.loc[gsrc[c] < 0, c] = np.nan
            sources['gama'] = (np.asarray(gsrc['X']), np.asarray(gsrc['Y']), gsrc)
        if isinstance(mag_df, pd.DataFrame):
            msrc = mag_df.copy()
            if noneg:
                for c in msrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(msrc[c]):
                        msrc.loc[msrc[c] < 0, c] = np.nan
            sources['mag'] = (np.asarray(msrc['X']), np.asarray(msrc['Y']), msrc)

        if not sources:
            continue

        # sempre cria todas as colunas solicitadas
        out = {'X': xu, 'Y': yu}
        for f in features:
            arr = None
            for src in _source_order_for_feature(f):
                if src not in sources:
                    continue
                x, y, df = sources[src]
                real_col = _find_source_column(df, f)
                if real_col is None:
                    continue
                vals = df[real_col].to_numpy()
                arr = interp_at(x, y, vals, xu, yu, algorithm=algo, extrapolate=True)
                break

            if arr is None:
                # fonte não encontrada: mantém coluna com NaN (SOM vai imputar)
                arr = np.full_like(xu, np.nan, dtype='float32')
            out[f] = arr

        quadricula[fid][out_name] = pd.DataFrame(out)

    return out_name

# ---------------- widgets ----------------
# seleção e carregamento
w_escala = W.Dropdown(options=ESCALAS, value='100k', description='Escala')
w_filtro = W.Text(placeholder='ex.: SF23_YB', description='Filtro')
w_ids    = W.SelectMultiple(options=(), rows=10, description='Folhas')
w_selall = W.ToggleButton(value=False, description='Selecionar tudo', icon='check')
w_clear  = W.Button(description='Limpar', icon='trash')

w_ext    = W.IntSlider(min=0, max=2000, step=100, value=600, description='extend_size')
w_gama   = W.Dropdown(options=['gama_line_1105','gama_line_1089','gama_1039','gama_3022'], value='gama_line_1105', description='Gama')
w_mag    = W.Dropdown(options=['mag_line_1105','mag_line_1089','mag_1039','mag_3022'], value='mag_line_1105', description='Mag')
w_load   = W.Button(description='Carregar brutos', button_style='success', icon='download')

# interpolação
w_feats_interp = W.SelectMultiple(
    options=['GMT','CTCOR','eTh','eU','KPERC','UTHRAZAO','UKRAZAO','THKRAZAO','MDT'],
    value=('GMT','CTCOR','eTh','eU','KPERC','MDT'),
    rows=8, description='Features (grid)'
)
w_psize  = W.IntSlider(min=50, max=1000, step=50, value=200, description='Pixel (m)')
w_algo   = W.Dropdown(options=[('Linear','linear'),('Cúbico','cubic')], value='linear', description='Algoritmo')
w_nonegI = W.Checkbox(value=False, description='Negativos→NaN (grid)')
w_interpolar = W.Button(description='Interpolar grade', icon='shuffle')

# preview
w_layers = W.SelectMultiple(options=(), rows=6, description='Camadas')
w_cols   = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=6, description='Colunas')
w_nonegP = W.Checkbox(value=False, description='Remover negativos (preview)')
w_refresh = W.Button(description='Atualizar', icon='refresh')
w_plot   = W.Button(description='Pré-visualizar', icon='eye')

# SOM — treino/teste
w_feats  = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=8, description='Features (SOM)')
w_sigma  = W.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.5, description='sigma')
w_iter   = W.IntSlider(min=500, max=30000, step=500, value=10000, description='max_iter')
w_seed   = W.IntSlider(min=0, max=9999, step=1, value=42, description='seed')
w_flip   = W.Checkbox(value=False, description='flip N-S no plot')

# treino com vários k
w_ks_train = W.SelectMultiple(
    options=tuple(range(3, 31)),  # 3..30
    value=(8, 12, 16),
    rows=8, description='k p/ treinar'
)
w_train  = W.Button(description='Treinar SOM(s)', button_style='primary', icon='play')

# teste/aplicação por subset de folhas
w_test_ids = W.SelectMultiple(options=(), rows=8, description='Folhas (teste)')
w_seltest  = W.ToggleButton(value=False, description='Selecionar todas (teste)', icon='check')
w_k_apply  = W.Dropdown(options=[], description='k (aplicar)')
w_apply    = W.Button(description='Aplicar/Testar', icon='check-circle')
w_evalall  = W.Button(description='Comparar Ks (métricas)', icon='bar-chart')
w_clear_models = W.Button(description='Limpar modelos', icon='trash')

# NOVO: boxplots
w_boxplots = W.Button(description='Boxplots por atributo', icon='bar-chart', button_style='')

# status + saída
w_datagrid_label = W.HTML(value="<b>Camada SOM:</b> <i>—</i>")
w_models_label   = W.HTML(value="<b>Modelos treinados:</b> <i>—</i>")
w_out    = W.Output()

# estado global
quadricula = {}
data_grid = None  # nome da camada interpolada para uso no SOM
som_store = {}    # {k: {'som':..., 'imp':..., 'sca':..., 'feats':..., 'layer':...}}
som_last_pred = None  # {'k':..., 'classes':..., 'metas':..., 'fids':..., 'feats':..., 'layer':...}
# === Anexa a UI do BDC ao painel esquerdo ===
left.children = tuple(list(left.children) + [
    W.HTML("<hr><b>Imagens de Satélite — BDC/INPE (pystac)</b>"),
    W.HBox([
        W.VBox([w_bdc_filter, w_bdc_list, w_bdc_cols]),
        W.VBox([w_bdc_date, w_bdc_cloud, w_bdc_limit, w_bdc_sort]),
    ]),
    W.HBox([w_bdc_search, w_bdc_prev, w_bdc_save]),
    # (itens e bandas + amostrar na grade entram no passo 3)
])

# ---------------- helpers SOM ----------------
def _rescan_from_quadricula():
    q = globals().get('quadricula', {})
    layers = _scan_layers_from_quadricula(q)
    w_layers.options = layers
    # tenta selecionar a camada interpolada atual, senão gama/mag, senão a primeira
    global data_grid
    pick = ()
    if data_grid and data_grid in layers:
        pick = (data_grid,)
    else:
        pref = [w_gama.value, w_mag.value]
        pick = tuple([p for p in pref if p in layers]) or (layers[:1] if layers else ())
    w_layers.value = pick

    cols = _available_columns(q, w_layers.value)
    if not cols:
        cols = ('MDT',)
    w_cols.options = cols
    w_cols.value = tuple([c for c in ('MDT',) if c in cols]) or (cols[0],)

    # features para SOM: apenas numéricas das camadas selecionadas
    numeric = []
    for fid, blob in q.items():
        for lay in w_layers.value:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in cols:
                    if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
                        numeric.append(c)
    opts = sorted(set(numeric), key=lambda c: (c!='MDT', c)) or ['MDT']
    w_feats.options = opts
    keep = [c for c in w_feats.value if c in opts] or (['MDT'] if 'MDT' in opts else opts[:min(5,len(opts))])
    w_feats.value = tuple(keep)

    # popula folhas disponíveis para TESTE: somente as que possuem a camada interpolada
    test_opts = []
    if data_grid:
        for fid, blob in q.items():
            if data_grid in blob and isinstance(blob[data_grid], pd.DataFrame):
                test_opts.append(fid)
    w_test_ids.options = tuple(sorted(test_opts))
    w_test_ids.value = tuple(sorted(test_opts))[:min(4, len(test_opts))]  # seleção inicial

    # k (aplicar) a partir dos modelos treinados disponíveis
    ks = sorted(list(som_store.keys()))
    w_k_apply.options = ks
    if ks:
        w_k_apply.value = ks[0]

def _normalize_xy(df):
    if not {'E_utm','N_utm'}.issubset(df.columns):
        if {'X','Y'}.issubset(df.columns):
            df = df.rename(columns={'X':'E_utm','Y':'N_utm'}).copy()
        else:
            raise ValueError("Camada sem colunas de coordenadas ('X','Y' ou 'E_utm','N_utm').")
    # ordena p/ reshape consistente: N decrescente, E crescente
    df = df.sort_values(['N_utm','E_utm'], ascending=[False, True], ignore_index=True, kind='mergesort')
    xs1d = np.sort(df['E_utm'].unique())
    ys1d = np.sort(df['N_utm'].unique())
    nx, ny = xs1d.size, ys1d.size
    xs_mesh, ys_mesh = np.meshgrid(xs1d, ys1d)  # (ny, nx)
    return df, xs_mesh, ys_mesh, nx, ny

def _build_matrix_for_fids(quadricula, features, layer, fids=None):
    fids_all = sorted(quadricula.keys()) if fids is None else list(fids)
    all_blocks, slc, metas = [], {}, {}
    k = 0
    for fid in fids_all:
        blob = quadricula.get(fid, {})
        if layer not in blob:
            continue
        df = blob[layer].copy()
        try:
            df, xs_mesh, ys_mesh, nx, ny = _normalize_xy(df)
        except Exception:
            continue
        metas[fid] = {'nx': nx, 'ny': ny, 'xs': xs_mesh, 'ys': ys_mesh}
        X = df[features].to_numpy(dtype='float32')
        if X.size == 0:
            continue
        all_blocks.append(X)
        slc[fid] = slice(k, k + len(X)); k += len(X)
    if not all_blocks:
        raise RuntimeError(f"Nenhuma folha com '{layer}' e as features escolhidas foi encontrada.")
    return np.vstack(all_blocks), slc, metas

def _qe(som, X_std):
    D = som.transform(X_std)              # distâncias a cada neurônio
    return float(np.mean(np.min(D, axis=1)))

def _te_1d(som, X_std):
    D = som.transform(X_std)
    bmu = np.argmin(D, axis=1)
    D2 = D.copy(); D2[np.arange(D.shape[0]), bmu] = np.inf
    sbmu = np.argmin(D2, axis=1)
    return float(np.mean(np.abs(bmu - sbmu) > 1))

def _plot_classes(classes_by_fid, metas, n_clusters, flip_ns=False, titulo='Mapa preditivo (SOM)'):
    cmap = _make_discrete_cmap(n_clusters)
    bounds = np.arange(-0.5, n_clusters + 0.5, 1)
    norm = BoundaryNorm(bounds, ncolors=n_clusters, clip=True)

    fig, ax = plt.subplots(figsize=(10, 10), facecolor='w')
    for fid in sorted(classes_by_fid.keys()):
        Z = classes_by_fid[fid]
        if flip_ns:
            Z = np.flipud(Z)
        xs = metas[fid]['xs']; ys = metas[fid]['ys']
        ax.pcolormesh(xs, ys, Z, cmap=cmap, norm=norm, shading='nearest', rasterized=True)

    ax.set_aspect('equal')
    ax.set_title(titulo)
    cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, ticks=np.arange(n_clusters), pad=0.01)
    cbar.ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)])
    cbar.set_label('Classes')
    plt.tight_layout()
    plt.show()

def _predict_per_folha(som, X_std, slc, metas):
    out = {}
    for fid, s in slc.items():
        y = som.predict(X_std[s])
        ny, nx = metas[fid]['ny'], metas[fid]['nx']
        out[fid] = y.reshape(ny, nx)
    return out

# ------- tabela longa + boxplots --------
def som_build_long_table(quadricula, layer, classes_by_fid, metas, atributos, fids=None):
    """
    Monta um DataFrame 'longo': colunas ['fid','E_utm','N_utm','classe', *atributos]
    alinhando a ordem do df com a ordem do reshape usada no SOM (N decresc, E cresc).
    """
    rows = []
    fids_iter = list(classes_by_fid.keys()) if fids is None else list(fids)
    for fid in fids_iter:
        if fid not in classes_by_fid:
            continue
        Z = classes_by_fid[fid]
        blob = quadricula.get(fid, {})
        if layer not in blob:
            continue
        df = blob[layer].copy()
        df, xs_mesh, ys_mesh, nx, ny = _normalize_xy(df)
        # sanity
        if Z.shape != (ny, nx):
            # tenta ajustar: achatar e recortar
            Zv = np.ravel(Z)[:ny*nx]
        else:
            Zv = Z.ravel(order='C')
        # prepara subset com atributos
        cols_keep = []
        for a in atributos:
            if a in df.columns:
                cols_keep.append(a)
        sub = pd.DataFrame({
            'fid': fid,
            'E_utm': df['E_utm'].to_numpy(),
            'N_utm': df['N_utm'].to_numpy(),
            'classe': Zv.astype(int)
        })
        for a in cols_keep:
            sub[a] = df[a].to_numpy()
        rows.append(sub)
    if not rows:
        raise RuntimeError("Nenhum dado disponível para montar a tabela longa (verifique layer/atributos).")
    out = pd.concat(rows, axis=0, ignore_index=True)
    return out
    
def plot_boxplots_por_atributo(
    df_long: pd.DataFrame,
    atributos: list[str],
    classes: list | None = None,
    ncols: int = 2,
    showfliers: bool = False,
    rotation: int = 45,
    sharey: bool = True,
    figsize_cell: tuple[float, float] = (4.0, 3.2),
    suptitle: str | None = None,
):
    """
    Cria uma grade de subplots: 1 subplot por classe.
      - Eixo X: atributos (categorias)
      - Caixas: distribuição do atributo naquela classe
    Parâmetros:
      df_long     : DataFrame com colunas ['classe','E_utm','N_utm', *atributos]
      atributos   : lista de colunas numéricas a plotar
      classes     : lista/ordem opcional das classes; se None, usa únicas ordenadas
      ncols       : número de colunas na grade de subplots
      showfliers  : mostra (True) ou omite (False) outliers
      rotation    : rotação dos rótulos no eixo X
      sharey      : compartilha eixo Y entre subplots
      figsize_cell: tamanho (largura, altura) de cada célula (subplot)
      suptitle    : título geral da figura (opcional)
    """
    import math
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pd

    if classes is None:
        classes = sorted(pd.Series(df_long['classe']).dropna().unique())

    n_classes = len(classes)
    if n_classes == 0:
        print("[Boxplot] Nenhuma classe para plotar.")
        return None

    ncols = max(1, int(ncols))
    nrows = math.ceil(n_classes / ncols)

    fig_w = max(6.0, figsize_cell[0] * ncols)
    fig_h = max(3.2, figsize_cell[1] * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharey=sharey)
    axes = np.atleast_1d(axes).ravel()

    # pré-coleta global (se sharey) para unificar limites do Y
    global_vals = []
    if sharey:
        for c in classes:
            sub = df_long[df_long['classe'] == c]
            for a in atributos:
                if a in sub.columns:
                    s = pd.to_numeric(sub[a], errors='coerce').dropna().values
                    if s.size:
                        global_vals.append(s)
        if global_vals:
            gcat = np.concatenate(global_vals)
            y_min, y_max = np.nanmin(gcat), np.nanmax(gcat)
            if not np.isfinite(y_min) or not np.isfinite(y_max) or y_min == y_max:
                y_min, y_max = None, None
        else:
            y_min, y_max = None, None
    else:
        y_min = y_max = None

    for i, c in enumerate(classes):
        ax = axes[i]
        sub = df_long[df_long['classe'] == c]

        vals_list, labels = [], []
        for a in atributos:
            if a not in sub.columns:
                continue
            s = pd.to_numeric(sub[a], errors='coerce').dropna()
            if s.size:
                vals_list.append(s.values)
                labels.append(a)

        # rótulo humano da classe (começando de 1, se for inteiro)
        try:
            c_label = int(c) + 1
        except Exception:
            c_label = c

        if not vals_list:
            ax.set_title(f"Classe {c_label} (sem dados)")
            ax.axis("off")
        else:
            ax.boxplot(vals_list, labels=labels, showfliers=showfliers)
            ax.set_title(f"Classe {c_label}")
            ax.set_xlabel("Atributo")
            ax.set_ylabel("Valor")
            ax.tick_params(axis='x', labelrotation=rotation)
            if y_min is not None and y_max is not None:
                # pequena margem
                pad = 0.03 * (y_max - y_min if y_max != y_min else 1.0)
                ax.set_ylim(y_min - pad, y_max + pad)

    # esconde eixos sobrando
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    if suptitle:
        fig.suptitle(suptitle)

    plt.tight_layout()
    plt.show()
    return fig

# ---------------- callbacks ----------------
def refresh_ids(*_):
    ids = _ids_from_mc(w_escala.value, w_filtro.value.strip() or None)
    w_ids.options = ids
    w_selall.value = False

def on_selall_change(ch):
    if ch['name'] == 'value':
        w_ids.value = tuple(w_ids.options) if ch['new'] else ()

def on_seltest_change(ch):
    if ch['name'] == 'value':
        w_test_ids.value = tuple(w_test_ids.options) if ch['new'] else ()

def on_clear_clicked(_):
    w_filtro.value = ''
    w_ids.value = ()

def on_load_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        print('# Montando grade…')
        quad = Build_mc(escala=w_escala.value, ID=list(w_ids.value), verbose=True)
        print('# Carregando dados brutos…')
        _g, _m = Upload_geof(
            quad,
            gama_xyz=w_gama.value,
            mag_xyz=w_mag.value,
            extend_size=int(w_ext.value)
        )
        quad = pop_nodata(quad)
        globals()['quadricula'] = quad
        print(f'Folhas ativas: {len(quad)}')
        globals()['data_grid'] = None
        w_datagrid_label.value = "<b>Camada SOM:</b> <i>— (interpole primeiro)</i>"
        # zera modelos e predições
        som_store.clear()
        globals()['som_last_pred'] = None
        w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        print('Pronto. Dados brutos anexados. Agora execute a INTERPOLAÇÃO.')

def on_refresh_clicked(_):
    with w_out:
        clear_output()
        if 'quadricula' not in globals():
            print('A variável global `quadricula` ainda não existe. Carregue dados primeiro.')
            return
        print('Re-escaneando `quadricula`…')
        _rescan_from_quadricula()
        print('Atualizado.')

def on_plot_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        if not w_layers.value:
            print('Nenhuma camada selecionada.')
            return
        q = globals().get('quadricula', {})
        for col in w_cols.value:
            _plot_layers_for_column(q, w_ids.value, w_layers.value, col, remove_negatives=w_nonegP.value)

def on_interpolar_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        feats_grid = list(w_feats_interp.value)
        if not feats_grid:
            print('Selecione ao menos 1 feature para a grade.')
            return
        q = globals().get('quadricula', {})
        if not q:
            print('Carregue os dados brutos primeiro.')
            return

        print(f"# Interpolando (algo={w_algo.value}, pixel={int(w_psize.value)} m)…")
        out_layer = _interpolate_current_selection(
            q, ids=w_ids.value,
            gama_key=w_gama.value, mag_key=w_mag.value,
            features=feats_grid, psize=int(w_psize.value),
            algo=w_algo.value, noneg=w_nonegI.value
        )
        globals()['quadricula'] = q
        globals()['data_grid'] = out_layer
        w_datagrid_label.value = f"<b>Camada SOM:</b> <code>{out_layer}</code>"
        print(f"→ Camada criada: {out_layer}")
        print("Dica: clique em 'Atualizar' e depois em 'Pré-visualizar' para conferir a grade.")
        # novos dados disponíveis ⇒ reset modelos/predições
        som_store.clear()
        globals()['som_last_pred'] = None
        w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        # foca a camada criada no preview
        if out_layer in w_layers.options:
            w_layers.value = (out_layer,)

def on_train_clicked(_):
    with w_out:
        clear_output()
        feats = list(w_feats.value)
        if not feats:
            print("Selecione ao menos 1 feature (SOM).")
            return
        if not globals().get('data_grid'):
            print("Interpole a grade primeiro (botão 'Interpolar grade').")
            return
        layer = globals()['data_grid']
        q = globals().get('quadricula', {})
        if not q:
            print('Carregue dados e interpele a grade antes do SOM.')
            return

        # Treina usando TODAS as folhas que possuam o layer interpolado (treino global)
        print(f"[TREINO] Montando matriz global a partir de '{layer}'…")
        X_all, _, _ = _build_matrix_for_fids(q, feats, layer, fids=None)

        # imput + scaler global (guardados junto do modelo)
        imp = SimpleImputer(strategy='median')
        X_imp = imp.fit_transform(X_all)
        sca   = StandardScaler().fit(X_imp)
        X_std = sca.transform(X_imp)

        ks = sorted(set(int(k) for k in w_ks_train.value))
        if not ks:
            print("Selecione ao menos um valor de k para treinar.")
            return

        np.random.seed(int(w_seed.value))
        trained = []
        for k in ks:
            print(f" - Treinando SOM(k={k}, sigma={float(w_sigma.value)}, it={int(w_iter.value)}) …")
            som = SOM(m=int(k), n=1, sigma=float(w_sigma.value), dim=len(feats), max_iter=int(w_iter.value))
            som.fit(X_std)
            som_store[k] = {'som': som, 'imp': imp, 'sca': sca, 'feats': feats, 'layer': layer}
            trained.append(k)

        if trained:
            w_models_label.value = f"<b>Modelos treinados:</b> {', '.join(map(str, sorted(som_store.keys())))}"
            # atualiza lista de k disponíveis para aplicar
            w_k_apply.options = sorted(list(som_store.keys()))
            w_k_apply.value = w_k_apply.options[0]
            print("Modelos treinados com sucesso.")
        else:
            print("Nenhum modelo foi treinado.")

def on_apply_clicked(_):
    with w_out:
        clear_output()
        if not som_store:
            print("Nenhum modelo treinado. Use 'Treinar SOM(s)' antes.")
            return
        if not w_test_ids.value:
            print("Selecione ao menos 1 folha para teste/aplicação.")
            return
        k = int(w_k_apply.value)
        model = som_store.get(k)
        if model is None:
            print(f"k={k} não encontrado entre os modelos treinados.")
            return
        feats = model['feats']; layer = model['layer']
        q = globals().get('quadricula', {})

        # monta matriz APENAS com as folhas selecionadas para teste
        print(f"[TESTE] Preparando subset ({len(w_test_ids.value)} folha(s)) com layer '{layer}'…")
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e:
            print(str(e)); return

        X_te_std = model['sca'].transform(model['imp'].transform(X_te))

        # métricas no subset
        qe = _qe(model['som'], X_te_std)
        te = _te_1d(model['som'], X_te_std)
        df_metrics = pd.DataFrame([{'k': k, 'QE_test': qe, 'TE_test': te}])
        print(df_metrics.to_string(index=False))

        # previsão e plot
        classes = _predict_per_folha(model['som'], X_te_std, slc_te, metas_te)
        _plot_classes(
            classes, metas_te, n_clusters=k, flip_ns=bool(w_flip.value),
            titulo=f"SOM (aplicar): k={k} | sigma={float(w_sigma.value)} | it={int(w_iter.value)} | layer={layer}"
        )

        # salvar estado para análises (boxplots etc.)
        globals()['som_last_pred'] = {
            'k': k, 'classes': classes, 'metas': metas_te,
            'fids': tuple(w_test_ids.value), 'feats': tuple(feats), 'layer': layer
        }
        print("[SOM] Predição salva: som_last_pred (k, classes, metas, fids, feats, layer).")

def on_evalall_clicked(_):
    with w_out:
        clear_output()
        if not som_store:
            print("Nenhum modelo treinado. Use 'Treinar SOM(s)' antes.")
            return
        if not w_test_ids.value:
            print("Selecione ao menos 1 folha para avaliação.")
            return
        # usamos as features/layer do primeiro modelo como referência
        any_k = next(iter(som_store))
        feats = som_store[any_k]['feats']; layer = som_store[any_k]['layer']
        q = globals().get('quadricula', {})
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e:
            print(str(e)); return
        rows = []
        for k in sorted(som_store.keys()):
            model = som_store[k]
            # sanity check: mesmas feats/layer
            if model['feats'] != feats or model['layer'] != layer:
                rows.append({'k': k, 'QE_test': np.nan, 'TE_test': np.nan, 'obs': 'incompatível (feats/layer)'})
                continue
            X_te_std = model['sca'].transform(model['imp'].transform(X_te))
            rows.append({'k': k, 'QE_test': _qe(model['som'], X_te_std), 'TE_test': _te_1d(model['som'], X_te_std)})
        df = pd.DataFrame(rows).sort_values('QE_test', ascending=True, na_position='last', ignore_index=True)
        print(df.to_string(index=False))

def on_clear_models_clicked(_):
    som_store.clear()
    globals()['som_last_pred'] = None
    w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
    w_k_apply.options = []
    with w_out:
        clear_output()
        print("Modelos apagados.")

def on_boxplots_clicked(_):
    with w_out:
        clear_output()
        if not som_store:
            print("Nenhum modelo treinado. Treine e aplique um SOM primeiro.")
            return
        # Preferir usar o último resultado aplicado
        lp = globals().get('som_last_pred')
        if lp is None:
            print("Nenhuma predição recente encontrada. Clique em 'Aplicar/Testar' e tente novamente.")
            return

        k = lp['k']; classes = lp['classes']; metas = lp['metas']
        fids = lp['fids']; feats = list(lp['feats']); layer = lp['layer']
        q = globals().get('quadricula', {})
        try:
            df_long = som_build_long_table(q, layer, classes, metas, atributos=feats, fids=fids)
        except RuntimeError as e:
            print(str(e)); return

        print(f"[Boxplots] {len(fids)} folha(s), k={k}, layer='{layer}', atributos={feats}")
        plot_boxplots_por_atributo(df_long, atributos=feats, ncols=2, showfliers=False)


# --- STAC / BDC ---
from pystac_client import Client
import requests, os
from urllib.parse import urlparse

BDC_ENDPOINT = "https://data.inpe.br/bdc/stac/v1"
def _aoi_bbox_from_ids(escala: str, ids: list[str]) -> list[float] | None:
    """
    Retorna bbox [minx, miny, maxx, maxy] em WGS84 (lon/lat) a partir dos ids de folhas.
    Usa import_malha_cartog(escala) para pegar a geometria oficial (não os pontos da geofísica).
    """
    if not ids:
        return None
    gdf = import_malha_cartog(escala=escala)
    gdf = gdf[gdf['id_folha'].astype(str).isin([str(i) for i in ids])].copy()
    if gdf.empty:
        return None
    if gdf.crs is None:
        # tenta a partir de uma coluna 'EPSG' (se existir); cai em 4674 (SIRGAS) -> 4326 como fallback
        try:
            epsg = int(gdf.get('EPSG').dropna().iloc[0])
            gdf = gdf.set_crs(epsg)
        except Exception:
            pass
    try:
        gdf = gdf.to_crs(4326)
    except Exception:
        pass
    minx, miny, maxx, maxy = gdf.total_bounds
    return [float(minx), float(miny), float(maxx), float(maxy)]

def _bdc_list_collections(pattern: str | None = None) -> list[str]:
    cli = Client.open(BDC_ENDPOINT)
    cols = [c.id for c in cli.get_collections()]
    if pattern:
        import re
        pat = re.compile(pattern, re.IGNORECASE)
        cols = [c for c in cols if pat.search(c)]
    return sorted(cols)

def _bdc_search_items(collections, bbox, dt_range: str, cloud_min: int, cloud_max: int, limit: int, sort_dir: str):
    """
    dt_range no formato 'YYYY-MM-DD/YYYY-MM-DD' ou 'YYYY-01-01/..'.
    cloud_min/max em 0..100.
    sort_dir: 'asc' (mais antigo primeiro) ou 'desc' (mais recente primeiro)
    """
    cli = Client.open(BDC_ENDPOINT)
    q = {"eo:cloud_cover": {"gte": int(cloud_min), "lte": int(cloud_max)}}
    # sortby segue STAC: campo 'properties.datetime' com prefixo '-' p/ desc
    sortby = ["properties.datetime"] if sort_dir == "asc" else ["-properties.datetime"]
    search = cli.search(collections=list(collections), bbox=bbox, datetime=dt_range, query=q, sortby=sortby, max_items=limit)
    return list(search.items())

def _bdc_pick_visual_asset(item) -> tuple[str, str] | None:
    """
    Tenta retornar (href, key) do asset 'visual' (RGB pronto). Fallback para 'thumbnail'.
    """
    # preferências comuns no BDC
    for key in ("visual", "overview", "thumbnail"):
        a = item.assets.get(key)
        if a and a.href:
            return a.href, key
    # em último caso, tenta montar RGB por bandas (não baixa aqui)
    for trip in (("red","green","blue"), ("B4","B3","B2"), ("B3","B2","B1")):
        if all(k in item.assets for k in trip):
            # devolve só o vermelho; o download/empilhamento pode ser feito depois se você quiser
            return item.assets[trip[0]].href, trip[0]
    return None

def _bdc_preview_thumbs(items, max_show=12):
    n = min(len(items), max_show)
    if n == 0:
        print("Nenhum item.")
        return
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 2.8*nrows))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        it = items[i]
        ax = axes[i]
        pair = _bdc_pick_visual_asset(it)
        ax.axis("off")
        ax.set_title(f"{it.collection_id}\n{it.datetime.date()}", fontsize=9)
        if pair is None:
            ax.text(0.5, 0.5, "sem preview", ha="center", va="center")
            continue
        href, key = pair
        # baixa thumbnail/overview para preview rápido na memória
        try:
            r = requests.get(href, timeout=15)
            r.raise_for_status()
            import PIL.Image as Image
            from io import BytesIO
            img = Image.open(BytesIO(r.content))
            ax.imshow(img)
        except Exception as e:
            ax.text(0.5, 0.5, f"erro preview\n{e}", ha="center", va="center", fontsize=8)
    # apaga sobras
    for j in range(i+1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

def _bdc_download_visual(items, outdir="satellite_bdc"):
    os.makedirs(outdir, exist_ok=True)
    saved = []
    for it in items:
        pair = _bdc_pick_visual_asset(it)
        if pair is None:
            continue
        href, key = pair
        # nome: <collection>_<id>_<key>.<ext>
        name = os.path.basename(urlparse(href).path)
        fname = f"{it.collection_id}_{it.id}_{key}_{name}"
        fpath = os.path.join(outdir, fname)
        try:
            if not os.path.exists(fpath):
                with requests.get(href, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    with open(fpath, "wb") as f:
                        for chunk in r.iter_content(chunk_size=1<<20):
                            if chunk:
                                f.write(chunk)
            saved.append(fpath)
        except Exception as e:
            print(f"[WARN] Falha ao baixar {href}: {e}")
    return saved
# --- Widgets BDC ---
w_bdc_filter = W.Text(placeholder='regex (ex.: landsat|sentinel|cbers)', description='Filtro')
w_bdc_list   = W.Button(description='Listar coleções', icon='list')
w_bdc_cols   = W.SelectMultiple(options=(), rows=8, description='Coleções')

w_bdc_date   = W.Text(value='2018-01-01/2025-12-31', description='Data (UTC)')
w_bdc_cloud  = W.IntRangeSlider(value=[0, 50], min=0, max=100, step=1, description='Nuvens (%)')
w_bdc_limit  = W.IntSlider(value=20, min=1, max=200, step=1, description='Limite')
w_bdc_sort   = W.Dropdown(options=[('Mais antigo','asc'), ('Mais recente','desc')], value='desc', description='Ordenar')

w_bdc_search = W.Button(description='Buscar itens', icon='search', button_style='info')
w_bdc_prev   = W.Button(description='Thumbnails', icon='image')
w_bdc_save   = W.Button(description='Baixar VISUAL', icon='download', button_style='success')

w_bdc_out    = W.Output()

# estado
bdc_items = []
def on_bdc_list_clicked(_):
    with w_bdc_out:
        clear_output()
        try:
            cols = _bdc_list_collections(w_bdc_filter.value.strip() or None)
            if not cols:
                print("Nenhuma coleção encontrada para o filtro.")
            else:
                w_bdc_cols.options = tuple(cols)
                print(f"{len(cols)} coleção(ões) listada(s). Selecione uma ou mais e pesquise.")
        except Exception as e:
            print("Erro ao listar coleções do BDC:", e)

def on_bdc_search_clicked(_):
    with w_bdc_out:
        clear_output()
        if not w_bdc_cols.value:
            print("Selecione ao menos 1 coleção.")
            return
        # usa a AOI das folhas selecionadas (w_ids)
        bbox = _aoi_bbox_from_ids(w_escala.value, list(w_ids.value))
        if not bbox:
            print("Selecione folhas (à esquerda) para definirmos a área.")
            return
        print("AOI (bbox WGS84):", bbox)
        try:
            items = _bdc_search_items(
                collections=w_bdc_cols.value,
                bbox=bbox,
                dt_range=w_bdc_date.value.strip(),
                cloud_min=w_bdc_cloud.value[0],
                cloud_max=w_bdc_cloud.value[1],
                limit=int(w_bdc_limit.value),
                sort_dir=w_bdc_sort.value
            )
        except Exception as e:
            print("Erro na busca STAC:", e)
            return
        globals()['bdc_items'] = items
        print(f"Encontrados {len(items)} item(ns). Use 'Thumbnails' para pré-visualizar ou 'Baixar VISUAL'.")

def on_bdc_prev_clicked(_):
    with w_bdc_out:
        clear_output()
        _bdc_preview_thumbs(globals().get('bdc_items', []), max_show=16)

def on_bdc_save_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items:
            print("Nenhuma lista de itens disponível. Faça a busca primeiro.")
            return
        out = _bdc_download_visual(items, outdir="satellite_bdc")
        if out:
            print(f"Arquivos salvos ({len(out)}):")
            for p in out:
                print(" -", p)
        else:
            print("Nenhum asset 'visual/overview/thumbnail' pôde ser baixado.")


        

# ligações
w_escala.observe(refresh_ids, names='value')
w_filtro.observe(refresh_ids, names='value')
w_selall.observe(on_selall_change, names='value')
w_seltest.observe(on_seltest_change, names='value')
w_clear.on_click(on_clear_clicked)

w_load.on_click(on_load_clicked)
w_refresh.on_click(on_refresh_clicked)
w_plot.on_click(on_plot_clicked)

w_interpolar.on_click(on_interpolar_clicked)

w_train.on_click(on_train_clicked)
w_apply.on_click(on_apply_clicked)
w_evalall.on_click(on_evalall_clicked)
w_clear_models.on_click(on_clear_models_clicked)
w_boxplots.on_click(on_boxplots_clicked)

# inicializa
refresh_ids()

# layout
left = W.VBox([
    W.HBox([w_escala, w_filtro]),
    W.HBox([
        w_ids,
        W.VBox([w_selall, w_clear, w_ext, w_gama, w_mag, w_load, w_refresh, w_plot]),
    ]),
    W.HTML("<hr><b>Interpolação para grade</b>"),
    W.HBox([w_feats_interp, W.VBox([w_psize, w_algo, w_nonegI, w_interpolar])]),
    w_datagrid_label
])

mid = W.VBox([
    W.HTML("<b>Pré-visualização</b>"),
    W.HBox([w_layers, w_cols]),
    w_nonegP,
    W.HTML("<hr><b>SOM — Treino</b>"),
    w_feats,
    W.HBox([w_sigma, w_iter, w_seed]),
    W.HBox([w_ks_train, w_train]),
    w_models_label
])

right = W.VBox([
    W.HTML("<b>SOM — Teste/Aplicação</b>"),
    W.HBox([w_test_ids, W.VBox([w_seltest, w_k_apply, w_apply, w_evalall, w_flip, w_clear_models, w_boxplots])])
])

ui = W.VBox([W.HBox([left, mid, right]), w_out])



#quadricula['SC23_ZA_IV']['geof_1089_linear'].head()

quadricula['SC23_ZA_IV']['geof_1089_linear'].describe().T


# -*- coding: utf-8 -*-
# Notebook único: Aerogeofísica (interp+SOM) + Imagens de Satélite (INPE/BDC STAC)
# Requer: pystac-client, rasterio, requests, pillow

# === Imports do seu projeto ===
from src import *
from verde_source import regular, interp_at

# === Terceiros ===
import os, re, json, requests, warnings
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import geopandas as gpd
import pyproj

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from matplotlib.colors import ListedColormap, BoundaryNorm

from sklearn_som.som import SOM
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import ipywidgets as W
from IPython.display import display, clear_output

import rasterio
from rasterio.enums import Resampling

from pystac_client import Client

warnings.filterwarnings("ignore")
%matplotlib inline

# =========================
# ======= UTIL GERAL ======
# =========================
ESCALAS = ['25k','50k','100k','250k','1kk']
BDC_ENDPOINT = "https://data.inpe.br/bdc/stac/v1"

def _norm_name(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(name).lower())

# Sinônimos -> feature canônica
_SYNONYMS = {
    'GMT'      : {'gmt','magigrf','magr','igrf','mag','gmtigrf','magig rf','magigrf'},
    'MDT'      : {'mdt','alte','altura'},
    'CTCOR'    : {'ctcor','ctc','ct'},
    'eTh'      : {'eth','eth_ppm','thc','th_ppm','ethppm','th'},
    'eU'       : {'eu','uc','u','euppm','u_ppm'},
    'KPERC'    : {'kperc','kc','k','kpct','k_percent'},
    'UTHRAZAO' : {'uthrazao','uratio','u_th','u/th','u_th_ratio'},
    'UKRAZAO'  : {'ukrazao','u_k','u/k','u_k_ratio'},
    'THKRAZAO' : {'thkrazao','th_k','th/k','th_k_ratio'},
}

def _find_source_column(df: pd.DataFrame, canonical: str) -> str | None:
    want = _norm_name(canonical)
    for c in df.columns:
        if _norm_name(c) == want:
            return c
    syns = _SYNONYMS.get(canonical, set())
    cols_norm = { _norm_name(c): c for c in df.columns }
    for s in syns:
        if s in cols_norm:
            return cols_norm[s]
    return None

def _source_order_for_feature(canonical: str) -> list[str]:
    if canonical in ('GMT', 'MDT'):
        return ['mag', 'gama']
    return ['gama', 'mag']

def _ids_from_mc(escala, filtro_regex=None):
    mc = import_malha_cartog(escala=escala)
    ids = mc['id_folha'].astype(str).tolist()
    if filtro_regex:
        pat = re.compile(filtro_regex, re.IGNORECASE)
        ids = [i for i in ids if pat.search(i)]
    return sorted(ids)

def _scan_layers_from_quadricula(q):
    layers = set()
    for fid, blob in (q or {}).items():
        for k, v in blob.items():
            if isinstance(v, pd.DataFrame):
                layers.add(k)
    return tuple(sorted(layers))

def _available_columns(q, layers):
    cols = set()
    for fid, blob in (q or {}).items():
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in df.columns:
                    if c in ('X','Y','E_utm','N_utm'):
                        continue
                    cols.add(c)
    cols = sorted(cols, key=lambda c: (c!='MDT', c))
    return tuple(cols)

def _global_min_max_numeric(q, ids, layers, column, remove_negatives=False):
    vals = []
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                s = df[column]
                if pd.api.types.is_numeric_dtype(s):
                    if remove_negatives:
                        s = s[s >= 0]
                    if s.size:
                        vals.append(s.to_numpy())
    if not vals:
        return None, None
    v = np.concatenate(vals)
    if v.size == 0 or np.all(np.isnan(v)):
        return None, None
    return float(np.nanmin(v)), float(np.nanmax(v))

def _plot_layers_for_column(q, ids, layers, column, remove_negatives=False):
    plt.figure(figsize=(12, 9))
    ax = plt.gca()
    is_numeric = False
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                is_numeric = pd.api.types.is_numeric_dtype(df[column])
                break
        if is_numeric: break

    if is_numeric:
        vmin, vmax = _global_min_max_numeric(q, ids, layers, column, remove_negatives)
        if vmin is not None and vmax is not None and vmin == vmax:
            eps = 1e-9
            vmin, vmax = vmin - eps, vmax + eps
        norm = colors.Normalize(vmin=vmin, vmax=vmax) if vmin is not None else None
        cmap = cm.get_cmap('terrain')

    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if not isinstance(df, pd.DataFrame) or column not in df.columns:
                continue
            d = df
            if is_numeric and remove_negatives:
                d = d[d[column] >= 0]
                if d.empty: continue
            if is_numeric:
                ax.scatter(d.X.values, d.Y.values, c=d[column].values,
                           s=0.1, cmap=cmap, norm=norm, marker='H')
            else:
                codes, _ = pd.factorize(d[column], sort=True)
                ax.scatter(d.X.values, d.Y.values, c=codes, s=0.1,
                           cmap='tab20', marker='H')

    ax.set_aspect('equal')
    ax.set_title(f'Pré-visualização • {column} • {len(ids)} folha(s) • camadas: {", ".join(layers)}')
    plt.axis('scaled')
    if is_numeric and norm is not None:
        cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
        vmin, vmax = norm.vmin, norm.vmax
        mid = (vmin + vmax) / 2.0
        cbar.set_ticks([vmin, mid, vmax])
        cbar.ax.set_yticklabels([f'{vmin:.3g}', f'{mid:.3g}', f'{vmax:.3g}'])
        cbar.set_label(f'{column} (min→máx)', rotation=90)
    plt.show()

def _make_discrete_cmap(n):
    base = plt.get_cmap('tab20')
    if hasattr(base, 'colors') and len(base.colors) >= n:
        return ListedColormap(base.colors[:n], name=f'tab20_{n}')
    return plt.get_cmap('nipy_spectral', n)

def _infer_suffix_from_names(*names):
    for nm in names:
        if not nm: continue
        m = re.search(r'(\d{4})', str(nm))
        if m: return m.group(1)
    return '0000'

# ===============================
# === Interpolação p/ a grade ===
# ===============================
def _interpolate_current_selection(quadricula, ids, gama_key, mag_key, features, psize, algo, noneg=False):
    suf = _infer_suffix_from_names(gama_key, mag_key)
    out_name = f"geof_{suf}_{algo}"

    for fid in ids:
        blob = quadricula.get(fid, {})
        gama_df = blob.get(gama_key)
        mag_df  = blob.get(mag_key)
        if (gama_df is None) and (mag_df is None): continue

        xu, yu = sintetic_grid(quadricula, fid, psize=int(psize))

        sources = {}
        if isinstance(gama_df, pd.DataFrame):
            gsrc = gama_df.copy()
            if noneg:
                for c in gsrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(gsrc[c]):
                        gsrc.loc[gsrc[c] < 0, c] = np.nan
            sources['gama'] = (np.asarray(gsrc['X']), np.asarray(gsrc['Y']), gsrc)
        if isinstance(mag_df, pd.DataFrame):
            msrc = mag_df.copy()
            if noneg:
                for c in msrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(msrc[c]):
                        msrc.loc[msrc[c] < 0, c] = np.nan
            sources['mag'] = (np.asarray(msrc['X']), np.asarray(msrc['Y']), msrc)

        if not sources: continue

        out = {'X': xu, 'Y': yu}
        for f in features:
            arr = None
            for src in _source_order_for_feature(f):
                if src not in sources: continue
                x, y, df = sources[src]
                real_col = _find_source_column(df, f)
                if real_col is None: continue
                vals = df[real_col].to_numpy()
                arr = interp_at(x, y, vals, xu, yu, algorithm=algo, extrapolate=True)
                break
            if arr is None:
                arr = np.full_like(xu, np.nan, dtype='float32')
            out[f] = arr

        quadricula[fid][out_name] = pd.DataFrame(out)

    return out_name

# =========================
# ======= SOM / plot ======
# =========================
def _normalize_xy(df):
    if not {'E_utm','N_utm'}.issubset(df.columns):
        if {'X','Y'}.issubset(df.columns):
            df = df.rename(columns={'X':'E_utm','Y':'N_utm'}).copy()
        else:
            raise ValueError("Camada sem colunas de coordenadas ('X','Y' ou 'E_utm','N_utm').")
    df = df.sort_values(['N_utm','E_utm'], ascending=[False, True], ignore_index=True, kind='mergesort')
    xs1d = np.sort(df['E_utm'].unique())
    ys1d = np.sort(df['N_utm'].unique())
    nx, ny = xs1d.size, ys1d.size
    xs_mesh, ys_mesh = np.meshgrid(xs1d, ys1d)  # (ny, nx)
    return df, xs_mesh, ys_mesh, nx, ny

def _build_matrix_for_fids(quadricula, features, layer, fids=None):
    fids_all = sorted(quadricula.keys()) if fids is None else list(fids)
    all_blocks, slc, metas = [], {}, {}
    k = 0
    for fid in fids_all:
        blob = quadricula.get(fid, {})
        if layer not in blob: continue
        df = blob[layer].copy()
        try:
            df, xs_mesh, ys_mesh, nx, ny = _normalize_xy(df)
        except Exception:
            continue
        metas[fid] = {'nx': nx, 'ny': ny, 'xs': xs_mesh, 'ys': ys_mesh}
        X = df[features].to_numpy(dtype='float32')
        if X.size == 0: continue
        all_blocks.append(X)
        slc[fid] = slice(k, k + len(X)); k += len(X)
    if not all_blocks:
        raise RuntimeError(f"Nenhuma folha com '{layer}' e as features escolhidas foi encontrada.")
    return np.vstack(all_blocks), slc, metas

def _qe(som, X_std):
    D = som.transform(X_std)
    return float(np.mean(np.min(D, axis=1)))

def _te_1d(som, X_std):
    D = som.transform(X_std)
    bmu = np.argmin(D, axis=1)
    D2 = D.copy(); D2[np.arange(D.shape[0]), bmu] = np.inf
    sbmu = np.argmin(D2, axis=1)
    return float(np.mean(np.abs(bmu - sbmu) > 1))

def _make_discrete_cmap(n):
    base = plt.get_cmap('tab20')
    if hasattr(base, 'colors') and len(base.colors) >= n:
        return ListedColormap(base.colors[:n], name=f'tab20_{n}')
    return plt.get_cmap('nipy_spectral', n)

def _plot_classes(classes_by_fid, metas, n_clusters, flip_ns=False, titulo='Mapa preditivo (SOM)'):
    cmap = _make_discrete_cmap(n_clusters)
    bounds = np.arange(-0.5, n_clusters + 0.5, 1)
    norm = BoundaryNorm(bounds, ncolors=n_clusters, clip=True)

    fig, ax = plt.subplots(figsize=(10, 10), facecolor='w')
    for fid in sorted(classes_by_fid.keys()):
        Z = classes_by_fid[fid]
        if flip_ns: Z = np.flipud(Z)
        xs = metas[fid]['xs']; ys = metas[fid]['ys']
        ax.pcolormesh(xs, ys, Z, cmap=cmap, norm=norm, shading='nearest', rasterized=True)

    ax.set_aspect('equal')
    ax.set_title(titulo)
    cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, ticks=np.arange(n_clusters), pad=0.01)
    cbar.ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)])
    cbar.set_label('Classes')
    plt.tight_layout(); plt.show()

def _predict_per_folha(som, X_std, slc, metas):
    out = {}
    for fid, s in slc.items():
        y = som.predict(X_std[s])
        ny, nx = metas[fid]['ny'], metas[fid]['nx']
        out[fid] = y.reshape(ny, nx)
    return out

# ------- tabela longa + boxplots --------
def som_build_long_table(quadricula, layer, classes_by_fid, metas, atributos, fids=None):
    rows = []
    fids_iter = list(classes_by_fid.keys()) if fids is None else list(fids)
    for fid in fids_iter:
        if fid not in classes_by_fid: continue
        Z = classes_by_fid[fid]
        blob = quadricula.get(fid, {})
        if layer not in blob: continue
        df = blob[layer].copy()
        df, xs_mesh, ys_mesh, nx, ny = _normalize_xy(df)
        if Z.shape != (ny, nx):
            Zv = np.ravel(Z)[:ny*nx]
        else:
            Zv = Z.ravel(order='C')
        cols_keep = [a for a in atributos if a in df.columns]
        sub = pd.DataFrame({
            'fid': fid,
            'E_utm': df['E_utm'].to_numpy(),
            'N_utm': df['N_utm'].to_numpy(),
            'classe': Zv.astype(int)
        })
        for a in cols_keep:
            sub[a] = df[a].to_numpy()
        rows.append(sub)
    if not rows:
        raise RuntimeError("Nenhum dado disponível para montar a tabela longa (verifique layer/atributos).")
    return pd.concat(rows, axis=0, ignore_index=True)

def plot_boxplots_por_atributo(
    df_long: pd.DataFrame, atributos: list[str], classes: list | None = None,
    ncols: int = 2, showfliers: bool = False, rotation: int = 45,
    sharey: bool = True, figsize_cell: tuple[float, float] = (4.0, 3.2), suptitle: str | None = None,
):
    import math
    if classes is None:
        classes = sorted(pd.Series(df_long['classe']).dropna().unique())
    n_classes = len(classes)
    if n_classes == 0:
        print("[Boxplot] Nenhuma classe para plotar."); return None
    ncols = max(1, int(ncols))
    nrows = math.ceil(n_classes / ncols)
    fig_w = max(6.0, figsize_cell[0] * ncols)
    fig_h = max(3.2, figsize_cell[1] * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharey=sharey)
    axes = np.atleast_1d(axes).ravel()

    global_vals = []
    if sharey:
        for c in classes:
            sub = df_long[df_long['classe'] == c]
            for a in atributos:
                if a in sub.columns:
                    s = pd.to_numeric(sub[a], errors='coerce').dropna().values
                    if s.size: global_vals.append(s)
        if global_vals:
            gcat = np.concatenate(global_vals)
            y_min, y_max = np.nanmin(gcat), np.nanmax(gcat)
            if not np.isfinite(y_min) or not np.isfinite(y_max) or y_min == y_max:
                y_min, y_max = None, None
        else:
            y_min = y_max = None
    else:
        y_min = y_max = None

    for i, c in enumerate(classes):
        ax = axes[i]
        sub = df_long[df_long['classe'] == c]
        vals_list, labels = [], []
        for a in atributos:
            if a not in sub.columns: continue
            s = pd.to_numeric(sub[a], errors='coerce').dropna()
            if s.size:
                vals_list.append(s.values); labels.append(a)
        try:
            c_label = int(c) + 1
        except Exception:
            c_label = c

        if not vals_list:
            ax.set_title(f"Classe {c_label} (sem dados)"); ax.axis("off")
        else:
            ax.boxplot(vals_list, labels=labels, showfliers=showfliers)
            ax.set_title(f"Classe {c_label}")
            ax.set_xlabel("Atributo"); ax.set_ylabel("Valor")
            ax.tick_params(axis='x', labelrotation=rotation)
            if y_min is not None and y_max is not None:
                pad = 0.03 * (y_max - y_min if y_max != y_min else 1.0)
                ax.set_ylim(y_min - pad, y_max + pad)

    for j in range(i + 1, len(axes)): axes[j].axis("off")
    if suptitle: fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()
    return fig

# ============================================
# ====== BLOCO BDC / STAC (Notebook) =========
# ============================================
def _aoi_bbox_from_ids(escala: str, ids: list[str]) -> list[float] | None:
    if not ids: return None
    gdf = import_malha_cartog(escala=escala)
    gdf = gdf[gdf['id_folha'].astype(str).isin([str(i) for i in ids])].copy()
    if gdf.empty: return None
    if gdf.crs is None:
        try:
            epsg = int(gdf.get('EPSG').dropna().iloc[0]); gdf = gdf.set_crs(epsg)
        except Exception:
            pass
    try:
        gdf = gdf.to_crs(4326)
    except Exception:
        pass
    minx, miny, maxx, maxy = gdf.total_bounds
    return [float(minx), float(miny), float(maxx), float(maxy)]

def _bdc_list_collections(pattern: str | None = None) -> list[str]:
    cli = Client.open(BDC_ENDPOINT)
    cols = [c.id for c in cli.get_collections()]
    if pattern:
        pat = re.compile(pattern, re.IGNORECASE)
        cols = [c for c in cols if pat.search(c)]
    return sorted(cols)

def _bdc_search_items(collections, bbox, dt_range: str, cloud_min: int, cloud_max: int, limit: int, sort_dir: str):
    cli = Client.open(BDC_ENDPOINT)
    q = {"eo:cloud_cover": {"gte": int(cloud_min), "lte": int(cloud_max)}}
    sortby = ["properties.datetime"] if sort_dir == "asc" else ["-properties.datetime"]
    search = cli.search(collections=list(collections), bbox=bbox, datetime=dt_range, query=q, sortby=sortby, max_items=limit)
    return list(search.items())

def _bdc_list_band_assets(item):
    # retorna dict de bandas GeoTIFF (nome -> href). Tenta canônicos
    out_all = {k: a.href for k, a in item.assets.items()
               if (a and a.href and (str(a.media_type or '').lower().find('tif')>=0
                                     or a.href.lower().endswith(('.tif','.tiff'))))}
    if not out_all: return {}
    alias = {
        'red'  : ('red','B4','BAND14','BAND3','B03','R'),
        'green': ('green','B3','BAND13','BAND2','B02','G'),
        'blue' : ('blue','B2','BAND12','BAND1','B01','B'),
        'nir'  : ('nir','B8','B08','BAND16','NIR'),
    }
    canon = {}
    for can, keys in alias.items():
        for k in keys:
            if k in item.assets and str(item.assets[k].href).lower().endswith(('.tif','.tiff')):
                canon[can] = item.assets[k].href; break
    return canon if canon else out_all

def _bdc_pick_visual_asset(item) -> tuple[str, str] | None:
    for key in ("visual", "overview", "thumbnail"):
        a = item.assets.get(key)
        if a and a.href: return a.href, key
    for trip in (("red","green","blue"), ("B4","B3","B2"), ("B3","B2","B1")):
        if all(k in item.assets for k in trip):
            return item.assets[trip[0]].href, trip[0]  # só p/ saber que há bandas
    return None

def _qrescale(b, p=(2,98)):
    b = np.asarray(b, dtype="float32")
    lo, hi = np.nanpercentile(b, p)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = np.nanmin(b), np.nanmax(b)
    b = np.clip((b - lo) / (hi - lo + 1e-9), 0, 1)
    return b

def _bdc_preview_thumbs(items, max_show=12):
    n = min(len(items), max_show)
    if n == 0:
        print("Nenhum item."); return
    ncols = 4; nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 2.8*nrows))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        it = items[i]
        ax = axes[i]; ax.axis("off")
        ax.set_title(f"{it.collection_id}\n{it.datetime.date()}", fontsize=9)
        pair = _bdc_pick_visual_asset(it)
        href = pair[0] if pair else None
        # 1) JPEG/PNG
        if href and href.lower().endswith((".jpg",".jpeg",".png")):
            try:
                r = requests.get(href, timeout=15); r.raise_for_status()
                from io import BytesIO
                import PIL.Image as Image
                ax.imshow(Image.open(BytesIO(r.content))); continue
            except Exception: pass
        # 2) Tenta COG/GeoTIFF rápido
        try:
            bmap = _bdc_list_band_assets(it)
            hrefs = None
            if {'red','green','blue'}.issubset(bmap):
                hrefs = [bmap['red'], bmap['green'], bmap['blue']]
            elif len(bmap) >= 3:
                hrefs = list(bmap.values())[:3]
            elif len(bmap) >= 1:
                hrefs = [list(bmap.values())[0]]
            if not hrefs:
                ax.text(0.5, 0.5, "sem preview", ha="center", va="center"); continue
            with rasterio.open(hrefs[0]) as src0:
                h = min(512, src0.height); w = min(512, src0.width)
                if len(hrefs) >= 3:
                    rgb = []
                    for href_b in hrefs[:3]:
                        with rasterio.open(href_b) as s:
                            b = s.read(1, out_shape=(h, w), resampling=Resampling.bilinear)
                            rgb.append(_qrescale(b))
                    ax.imshow(np.dstack(rgb))
                else:
                    b = src0.read(1, out_shape=(h, w), resampling=Resampling.bilinear)
                    ax.imshow(_qrescale(b), cmap="gray")
        except Exception as e:
            ax.text(0.5, 0.5, f"erro\n{e}", ha="center", va="center", fontsize=8)
    for j in range(i+1, len(axes)): axes[j].axis("off")
    plt.tight_layout(); plt.show()

def _grid_epsg_for_fid(escala, fid):
    try:
        g = import_malha_cartog(escala=escala)
        epsg = int(g.loc[g['id_folha'].astype(str)==str(fid), 'EPSG'].iloc[0])
        return epsg
    except Exception:
        return None

def _sample_to_grid(src, xs, ys, src_crs, dst_epsg, chunk=50000):
    tr = pyproj.Transformer.from_crs(src_crs, pyproj.CRS.from_epsg(int(dst_epsg)), always_xy=True).inverse
    # invertido: queremos lon/lat (ou CRS do raster) a partir de UTM -> então usamos inverse acima
    vals = np.empty(xs.size, dtype='float32')
    for i in range(0, xs.size, chunk):
        j = i + chunk
        # Converter (E_utm, N_utm) -> CRS do raster
        xdst = xs[i:j]; ydst = ys[i:j]
        px, py = tr(xdst, ydst)  # dst_epsg -> src_crs
        smp = np.array(list(src.sample(zip(px, py))))
        vals[i:j] = smp[:, 0] if smp.ndim == 2 else smp
    return vals

# =========================
# ======= WIDGETS UI ======
# =========================
# Seleção e carregamento
w_escala = W.Dropdown(options=ESCALAS, value='100k', description='Escala')
w_filtro = W.Text(placeholder='ex.: SF23_YB', description='Filtro')
w_ids    = W.SelectMultiple(options=(), rows=10, description='Folhas')
w_selall = W.ToggleButton(value=False, description='Selecionar tudo', icon='check')
w_clear  = W.Button(description='Limpar', icon='trash')

w_ext    = W.IntSlider(min=0, max=2000, step=100, value=600, description='extend_size')
w_gama   = W.Dropdown(options=['gama_line_1105','gama_line_1089','gama_1039','gama_3022'], value='gama_line_1105', description='Gama')
w_mag    = W.Dropdown(options=['mag_line_1105','mag_line_1089','mag_1039','mag_3022'], value='mag_line_1105', description='Mag')
w_load   = W.Button(description='Carregar brutos', button_style='success', icon='download')

# Interpolação
w_feats_interp = W.SelectMultiple(
    options=['GMT','CTCOR','eTh','eU','KPERC','UTHRAZAO','UKRAZAO','THKRAZAO','MDT'],
    value=('GMT','CTCOR','eTh','eU','KPERC','MDT'),
    rows=8, description='Features (grid)'
)
w_psize  = W.IntSlider(min=50, max=1000, step=50, value=200, description='Pixel (m)')
w_algo   = W.Dropdown(options=[('Linear','linear'),('Cúbico','cubic')], value='linear', description='Algoritmo')
w_nonegI = W.Checkbox(value=False, description='Negativos→NaN (grid)')
w_interpolar = W.Button(description='Interpolar grade', icon='shuffle')

# Preview
w_layers = W.SelectMultiple(options=(), rows=6, description='Camadas')
w_cols   = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=6, description='Colunas')
w_nonegP = W.Checkbox(value=False, description='Remover negativos (preview)')
w_refresh = W.Button(description='Atualizar', icon='refresh')
w_plot   = W.Button(description='Pré-visualizar', icon='eye')

# SOM — treino/teste
w_feats  = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=8, description='Features (SOM)')
w_sigma  = W.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.5, description='sigma')
w_iter   = W.IntSlider(min=500, max=30000, step=500, value=10000, description='max_iter')
w_seed   = W.IntSlider(min=0, max=9999, step=1, value=42, description='seed')
w_flip   = W.Checkbox(value=False, description='flip N-S no plot')

w_ks_train = W.SelectMultiple(options=tuple(range(3, 31)), value=(8, 12, 16), rows=8, description='k p/ treinar')
w_train  = W.Button(description='Treinar SOM(s)', button_style='primary', icon='play')

# Teste/aplicação
w_test_ids = W.SelectMultiple(options=(), rows=8, description='Folhas (teste)')
w_seltest  = W.ToggleButton(value=False, description='Selecionar todas (teste)', icon='check')
w_k_apply  = W.Dropdown(options=[], description='k (aplicar)')
w_apply    = W.Button(description='Aplicar/Testar', icon='check-circle')
w_evalall  = W.Button(description='Comparar Ks (métricas)', icon='bar-chart')
w_clear_models = W.Button(description='Limpar modelos', icon='trash')
w_boxplots = W.Button(description='Boxplots por atributo', icon='bar-chart', button_style='')

# BDC — STAC
w_bdc_filter = W.Text(placeholder='regex (ex.: landsat|sentinel|cbers)', description='Filtro')
w_bdc_list   = W.Button(description='Listar coleções', icon='list')
w_bdc_cols   = W.SelectMultiple(options=(), rows=8, description='Coleções')

w_bdc_date   = W.Text(value='2018-01-01/2025-12-31', description='Data (UTC)')
w_bdc_cloud  = W.IntRangeSlider(value=[0, 50], min=0, max=100, step=1, description='Nuvens (%)')
w_bdc_limit  = W.IntSlider(value=20, min=1, max=200, step=1, description='Limite')
w_bdc_sort   = W.Dropdown(options=[('Mais antigo','asc'), ('Mais recente','desc')], value='desc', description='Ordenar')

w_bdc_search = W.Button(description='Buscar itens', icon='search', button_style='info')
w_bdc_prev   = W.Button(description='Thumbnails', icon='image')
w_bdc_save   = W.Button(description='Baixar VISUAL', icon='download', button_style='success')

# seleção de item/bandas e amostragem p/ grade
w_bdc_item   = W.Dropdown(options=[], description='Item')
w_bdc_bands  = W.SelectMultiple(options=(), rows=6, description='Bandas')
w_bdc_prefix = W.Text(value='sat', description='Prefixo')
w_bdc_togrid = W.Button(description='Amostrar p/ grade', icon='link', button_style='warning')

# status + saída
w_datagrid_label = W.HTML(value="<b>Camada SOM:</b> <i>—</i>")
w_models_label   = W.HTML(value="<b>Modelos treinados:</b> <i>—</i>")
w_out    = W.Output()
w_bdc_out= W.Output()

# =========================
# ====== ESTADO GLOBAL ====
# =========================
quadricula = {}
data_grid = None
som_store = {}
som_last_pred = None
bdc_items = []

# =========================
# ====== CALLBACKS UI =====
# =========================
def refresh_ids(*_):
    ids = _ids_from_mc(w_escala.value, w_filtro.value.strip() or None)
    w_ids.options = ids
    w_selall.value = False

def on_selall_change(ch):
    if ch['name'] == 'value':
        w_ids.value = tuple(w_ids.options) if ch['new'] else ()

def on_seltest_change(ch):
    if ch['name'] == 'value':
        w_test_ids.value = tuple(w_test_ids.options) if ch['new'] else ()

def on_clear_clicked(_):
    w_filtro.value = ''; w_ids.value = ()

def on_load_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.'); return
        print('# Montando grade…')
        quad = Build_mc(escala=w_escala.value, ID=list(w_ids.value), verbose=True)
        print('# Carregando dados brutos…')
        _g, _m = Upload_geof(quad, gama_xyz=w_gama.value, mag_xyz=w_mag.value, extend_size=int(w_ext.value))
        quad = pop_nodata(quad)
        globals()['quadricula'] = quad
        print(f'Folhas ativas: {len(quad)}')
        globals()['data_grid'] = None
        w_datagrid_label.value = "<b>Camada SOM:</b> <i>— (interpole primeiro)</i>"
        som_store.clear(); globals()['som_last_pred'] = None
        w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        print('Pronto. Dados brutos anexados. Agora execute a INTERPOLAÇÃO.')

def _rescan_from_quadricula():
    q = globals().get('quadricula', {})
    layers = _scan_layers_from_quadricula(q)
    w_layers.options = layers
    global data_grid
    pick = ()
    if data_grid and data_grid in layers:
        pick = (data_grid,)
    else:
        pref = [w_gama.value, w_mag.value]
        pick = tuple([p for p in pref if p in layers]) or (layers[:1] if layers else ())
    w_layers.value = pick

    cols = _available_columns(q, w_layers.value) or ('MDT',)
    w_cols.options = cols
    w_cols.value = tuple([c for c in ('MDT',) if c in cols]) or (cols[0],)

    # features numéricas
    numeric = []
    for fid, blob in q.items():
        for lay in w_layers.value:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in cols:
                    if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
                        numeric.append(c)
    opts = sorted(set(numeric), key=lambda c: (c!='MDT', c)) or ['MDT']
    w_feats.options = opts
    keep = [c for c in w_feats.value if c in opts] or (['MDT'] if 'MDT' in opts else opts[:min(5,len(opts))])
    w_feats.value = tuple(keep)

    # folhas com layer interpolado
    test_opts = []
    if data_grid:
        for fid, blob in q.items():
            if data_grid in blob and isinstance(blob[data_grid], pd.DataFrame):
                test_opts.append(fid)
    w_test_ids.options = tuple(sorted(test_opts))
    w_test_ids.value = tuple(sorted(test_opts))[:min(4, len(test_opts))]
    ks = sorted(list(som_store.keys()))
    w_k_apply.options = ks
    if ks: w_k_apply.value = ks[0]

def on_refresh_clicked(_):
    with w_out:
        clear_output()
        if 'quadricula' not in globals():
            print('Carregue dados primeiro.'); return
        print('Re-escaneando `quadricula`…')
        _rescan_from_quadricula()
        print('Atualizado.')

def on_plot_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value: print('Selecione ao menos 1 folha.'); return
        if not w_layers.value: print('Nenhuma camada selecionada.'); return
        q = globals().get('quadricula', {})
        for col in w_cols.value:
            _plot_layers_for_column(q, w_ids.value, w_layers.value, col, remove_negatives=w_nonegP.value)

def on_interpolar_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value: print('Selecione ao menos 1 folha.'); return
        feats_grid = list(w_feats_interp.value)
        if not feats_grid: print('Selecione ao menos 1 feature para a grade.'); return
        q = globals().get('quadricula', {})
        if not q: print('Carregue os dados brutos primeiro.'); return

        print(f"# Interpolando (algo={w_algo.value}, pixel={int(w_psize.value)} m)…")
        out_layer = _interpolate_current_selection(
            q, ids=w_ids.value, gama_key=w_gama.value, mag_key=w_mag.value,
            features=feats_grid, psize=int(w_psize.value), algo=w_algo.value, noneg=w_nonegI.value
        )
        globals()['quadricula'] = q
        globals()['data_grid'] = out_layer
        w_datagrid_label.value = f"<b>Camada SOM:</b> <code>{out_layer}</code>"
        print(f"→ Camada criada: {out_layer}")
        som_store.clear(); globals()['som_last_pred'] = None
        w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        if out_layer in w_layers.options:
            w_layers.value = (out_layer,)

def on_train_clicked(_):
    with w_out:
        clear_output()
        feats = list(w_feats.value)
        if not feats: print("Selecione ao menos 1 feature (SOM)."); return
        if not globals().get('data_grid'): print("Interpole a grade primeiro."); return
        layer = globals()['data_grid']
        q = globals().get('quadricula', {})
        if not q: print('Carregue dados e interpele a grade antes do SOM.'); return
        print(f"[TREINO] Montando matriz global a partir de '{layer}'…")
        X_all, _, _ = _build_matrix_for_fids(q, feats, layer, fids=None)
        imp = SimpleImputer(strategy='median')
        X_imp = imp.fit_transform(X_all)
        sca   = StandardScaler().fit(X_imp)
        X_std = sca.transform(X_imp)
        ks = sorted(set(int(k) for k in w_ks_train.value))
        if not ks: print("Selecione ao menos um k."); return
        np.random.seed(int(w_seed.value))
        trained = []
        for k in ks:
            print(f" - Treinando SOM(k={k}, sigma={float(w_sigma.value)}, it={int(w_iter.value)}) …")
            som = SOM(m=int(k), n=1, sigma=float(w_sigma.value), dim=len(feats), max_iter=int(w_iter.value))
            som.fit(X_std)
            som_store[k] = {'som': som, 'imp': imp, 'sca': sca, 'feats': feats, 'layer': layer}
            trained.append(k)
        if trained:
            w_models_label.value = f"<b>Modelos treinados:</b> {', '.join(map(str, sorted(som_store.keys())))}"
            w_k_apply.options = sorted(list(som_store.keys()))
            w_k_apply.value = w_k_apply.options[0]
            print("Modelos treinados com sucesso.")
        else:
            print("Nenhum modelo foi treinado.")

def on_apply_clicked(_):
    with w_out:
        clear_output()
        if not som_store: print("Nenhum modelo treinado."); return
        if not w_test_ids.value: print("Selecione ao menos 1 folha para teste."); return
        k = int(w_k_apply.value)
        model = som_store.get(k)
        if model is None: print(f"k={k} não encontrado."); return
        feats = model['feats']; layer = model['layer']
        q = globals().get('quadricula', {})
        print(f"[TESTE] Preparando subset ({len(w_test_ids.value)} folha(s)) com layer '{layer}'…")
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e:
            print(str(e)); return
        X_te_std = model['sca'].transform(model['imp'].transform(X_te))
        qe = _qe(model['som'], X_te_std); te = _te_1d(model['som'], X_te_std)
        df_metrics = pd.DataFrame([{'k': k, 'QE_test': qe, 'TE_test': te}])
        print(df_metrics.to_string(index=False))
        classes = _predict_per_folha(model['som'], X_te_std, slc_te, metas_te)
        _plot_classes(classes, metas_te, n_clusters=k, flip_ns=bool(w_flip.value),
                      titulo=f"SOM (aplicar): k={k} | sigma={float(w_sigma.value)} | it={int(w_iter.value)} | layer={layer}")
        globals()['som_last_pred'] = {
            'k': k, 'classes': classes, 'metas': metas_te,
            'fids': tuple(w_test_ids.value), 'feats': tuple(feats), 'layer': layer
        }
        print("[SOM] Predição salva: som_last_pred.")

def on_evalall_clicked(_):
    with w_out:
        clear_output()
        if not som_store: print("Nenhum modelo treinado."); return
        if not w_test_ids.value: print("Selecione ao menos 1 folha."); return
        any_k = next(iter(som_store))
        feats = som_store[any_k]['feats']; layer = som_store[any_k]['layer']
        q = globals().get('quadricula', {})
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e:
            print(str(e)); return
        rows = []
        for k in sorted(som_store.keys()):
            model = som_store[k]
            if model['feats'] != feats or model['layer'] != layer:
                rows.append({'k': k, 'QE_test': np.nan, 'TE_test': np.nan, 'obs': 'incompatível'}); continue
            X_te_std = model['sca'].transform(model['imp'].transform(X_te))
            rows.append({'k': k, 'QE_test': _qe(model['som'], X_te_std), 'TE_test': _te_1d(model['som'], X_te_std)})
        df = pd.DataFrame(rows).sort_values('QE_test', ascending=True, na_position='last', ignore_index=True)
        print(df.to_string(index=False))

def on_clear_models_clicked(_):
    som_store.clear(); globals()['som_last_pred'] = None
    w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
    w_k_apply.options = []
    with w_out:
        clear_output(); print("Modelos apagados.")

def on_boxplots_clicked(_):
    with w_out:
        clear_output()
        if not som_store: print("Nenhum modelo treinado."); return
        lp = globals().get('som_last_pred')
        if lp is None: print("Nenhuma predição recente. Clique em 'Aplicar/Testar'."); return
        k = lp['k']; classes = lp['classes']; metas = lp['metas']
        fids = lp['fids']; feats = list(lp['feats']); layer = lp['layer']
        q = globals().get('quadricula', {})
        try:
            df_long = som_build_long_table(q, layer, classes, metas, atributos=feats, fids=fids)
        except RuntimeError as e:
            print(str(e)); return
        print(f"[Boxplots] {len(fids)} folha(s), k={k}, layer='{layer}', atributos={feats}")
        plot_boxplots_por_atributo(df_long, atributos=feats, ncols=2, showfliers=False)

# ====== BDC: callbacks ======
def on_bdc_list_clicked(_):
    with w_bdc_out:
        clear_output()
        try:
            cols = _bdc_list_collections(w_bdc_filter.value.strip() or None)
            if not cols:
                print("Nenhuma coleção encontrada para o filtro.")
            else:
                w_bdc_cols.options = tuple(cols)
                print(f"{len(cols)} coleção(ões) listada(s). Selecione e pesquise.")
        except Exception as e:
            print("Erro ao listar coleções do BDC:", e)

def _bdc_refresh_item_dropdown():
    items = globals().get('bdc_items', [])
    if not items:
        w_bdc_item.options = []; return
    opts = []
    for i, it in enumerate(items):
        lbl = f"{i:02d} | {it.collection_id} | {it.id} | {it.datetime.date()}"
        opts.append((lbl, i))
    w_bdc_item.options = opts
    w_bdc_item.value = opts[0][1] if opts else None

def on_bdc_search_clicked(_):
    with w_bdc_out:
        clear_output()
        if not w_bdc_cols.value:
            print("Selecione ao menos 1 coleção."); return
        bbox = _aoi_bbox_from_ids(w_escala.value, list(w_ids.value))
        if not bbox:
            print("Selecione folhas (painel esquerdo) para definirmos a área."); return
        print("AOI (bbox WGS84):", bbox)
        try:
            items = _bdc_search_items(
                collections=w_bdc_cols.value, bbox=bbox,
                dt_range=w_bdc_date.value.strip(),
                cloud_min=w_bdc_cloud.value[0], cloud_max=w_bdc_cloud.value[1],
                limit=int(w_bdc_limit.value), sort_dir=w_bdc_sort.value
            )
        except Exception as e:
            print("Erro na busca STAC:", e); return
        globals()['bdc_items'] = items
        print(f"Encontrados {len(items)} item(ns). Use 'Thumbnails' ou selecione um Item para amostrar na grade.")
        _bdc_refresh_item_dropdown()

def on_bdc_prev_clicked(_):
    with w_bdc_out:
        clear_output()
        _bdc_preview_thumbs(globals().get('bdc_items', []), max_show=16)

def _bdc_download_visual(items, outdir="satellite_bdc"):
    os.makedirs(outdir, exist_ok=True)
    saved = []
    for it in items:
        pair = _bdc_pick_visual_asset(it)
        if pair is None: continue
        href, key = pair
        name = os.path.basename(urlparse(href).path)
        fname = f"{it.collection_id}_{it.id}_{key}_{name}"
        fpath = os.path.join(outdir, fname)
        try:
            if not os.path.exists(fpath):
                with requests.get(href, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    with open(fpath, "wb") as f:
                        for chunk in r.iter_content(chunk_size=1<<20):
                            if chunk: f.write(chunk)
            saved.append(fpath)
        except Exception as e:
            print(f"[WARN] Falha ao baixar {href}: {e}")
    return saved

def on_bdc_save_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items:
            print("Nenhuma lista de itens. Faça a busca primeiro."); return
        out = _bdc_download_visual(items, outdir="satellite_bdc")
        if out:
            print(f"Arquivos salvos ({len(out)}):")
            for p in out: print(" -", p)
        else:
            print("Nenhum asset 'visual/overview/thumbnail' pôde ser baixado.")

def on_bdc_item_change(ch):
    if ch['name'] != 'value' or ch['new'] is None: return
    items = globals().get('bdc_items', [])
    if not items: return
    it = items[int(ch['new'])]
    bmap = _bdc_list_band_assets(it)
    bands = tuple(sorted(bmap.keys()))
    w_bdc_bands.options = bands
    if {'red','green','blue'}.issubset(bands):
        w_bdc_bands.value = ('red','green','blue')
    else:
        w_bdc_bands.value = bands[:min(3, len(bands))]

def on_bdc_togrid_clicked(_):
    with w_bdc_out:
        clear_output()
        if not globals().get('data_grid'):
            print("Interpole a grade primeiro."); return
        items = globals().get('bdc_items', [])
        if not items or w_bdc_item.value is None:
            print("Nenhum item selecionado."); return
        it = items[int(w_bdc_item.value)]
        bmap = _bdc_list_band_assets(it)
        pick = [b for b in w_bdc_bands.value if b in bmap]
        if not pick:
            print("Selecione ao menos 1 banda GeoTIFF."); return

        fids = list(w_ids.value) if w_ids.value else list(quadricula.keys())
        layer = globals()['data_grid']
        print(f"Amostrando {pick} → layer '{layer}' em {len(fids)} folha(s)…")

        n_written = 0
        for fid in fids:
            blob = quadricula.get(fid, {})
            if layer not in blob or not isinstance(blob[layer], pd.DataFrame):
                continue
            df = blob[layer]
            if not {'X','Y'}.issubset(df.columns) and not {'E_utm','N_utm'}.issubset(df.columns):
                continue
            # coordenadas
            if {'E_utm','N_utm'}.issubset(df.columns):
                xs, ys = df['E_utm'].to_numpy(), df['N_utm'].to_numpy()
            else:
                xs, ys = df['X'].to_numpy(), df['Y'].to_numpy()
            # epsg folha
            epsg = _grid_epsg_for_fid(w_escala.value, fid)
            if epsg is None:
                print(f"  - {fid}: EPSG desconhecido; pulando.")
                continue

            # amostrar cada banda
            for b in pick:
                href = bmap[b]
                try:
                    with rasterio.open(href) as src:
                        src_crs = src.crs
                        vals = _sample_to_grid(src, xs, ys, src_crs, epsg)
                        col = f"{w_bdc_prefix.value.strip() or 'sat'}_{b}"
                        df[col] = vals.astype('float32')
                        n_written += 1
                except Exception as e:
                    print(f"  - {fid}:{b} erro → {e}")

        if n_written:
            print(f"OK: {n_written} coluna(s) adicionada(s) nas folhas.")
            # atualiza listas/feats
            _rescan_from_quadricula()
            print("Colunas disponíveis atualizadas. As novas 'sat_*' já podem entrar no SOM.")
        else:
            print("Nenhuma coluna foi criada (verifique EPSG/GeoTIFF).")

# ============== Ligações ==============
w_escala.observe(refresh_ids, names='value')
w_filtro.observe(refresh_ids, names='value')
w_selall.observe(on_selall_change, names='value')
w_seltest.observe(on_seltest_change, names='value')
w_clear.on_click(on_clear_clicked)

w_load.on_click(on_load_clicked)
w_refresh.on_click(on_refresh_clicked)
w_plot.on_click(on_plot_clicked)

w_interpolar.on_click(on_interpolar_clicked)

w_train.on_click(on_train_clicked)
w_apply.on_click(on_apply_clicked)
w_evalall.on_click(on_evalall_clicked)
w_clear_models.on_click(on_clear_models_clicked)
w_boxplots.on_click(on_boxplots_clicked)

w_bdc_list.on_click(on_bdc_list_clicked)
w_bdc_search.on_click(on_bdc_search_clicked)
w_bdc_prev.on_click(on_bdc_prev_clicked)
w_bdc_save.on_click(on_bdc_save_clicked)
w_bdc_item.observe(on_bdc_item_change, names='value')
w_bdc_togrid.on_click(on_bdc_togrid_clicked)

# ============== Layout ==============
refresh_ids()

left = W.VBox([
    W.HBox([w_escala, w_filtro]),
    W.HBox([
        w_ids,
        W.VBox([w_selall, w_clear, w_ext, w_gama, w_mag, w_load, w_refresh, w_plot]),
    ]),
    W.HTML("<hr><b>Interpolação para grade</b>"),
    W.HBox([w_feats_interp, W.VBox([w_psize, w_algo, w_nonegI, w_interpolar])]),
    w_datagrid_label,

    W.HTML("<hr><b>Imagens de Satélite — BDC/INPE (STAC)</b>"),
    W.HBox([
        W.VBox([w_bdc_filter, w_bdc_list, w_bdc_cols]),
        W.VBox([w_bdc_date, w_bdc_cloud, w_bdc_limit, w_bdc_sort]),
    ]),
    W.HBox([w_bdc_search, w_bdc_prev, w_bdc_save]),
    W.HBox([w_bdc_item, w_bdc_bands]),
    W.HBox([w_bdc_prefix, w_bdc_togrid]),
    w_bdc_out
])

mid = W.VBox([
    W.HTML("<b>Pré-visualização</b>"),
    W.HBox([w_layers, w_cols]),
    w_nonegP,
    W.HTML("<hr><b>SOM — Treino</b>"),
    w_feats,
    W.HBox([w_sigma, w_iter, w_seed]),
    W.HBox([w_ks_train, w_train]),
    w_models_label
])

right = W.VBox([
    W.HTML("<b>SOM — Teste/Aplicação</b>"),
    W.HBox([w_test_ids, W.VBox([w_seltest, w_k_apply, w_apply, w_evalall, w_flip, w_clear_models, w_boxplots])])
])

ui = W.VBox([W.HBox([left, mid, right]), w_out])

display(ui)



# -*- coding: utf-8 -*-

# === UI integrada: Aerogeofísica + SOM + Satélite (BDC/INPE/STAC) ===

# --- imports base do seu projeto ---
from src import *                               # Build_mc, Upload_geof, pop_nodata, sintetic_grid, import_malha_cartog
from verde_source import regular, interp_at     # opcional; interp_at já utilizado
import verde as vd

from tqdm import tqdm
from pylab import *

import geopandas as gpd
import pyproj
import os, re, json, types, importlib, warnings, requests
from shapely.ops import transform as shp_transform
from shapely.geometry import Point, Polygon

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from matplotlib.colors import ListedColormap, BoundaryNorm

from sklearn_som.som import SOM
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import ipywidgets as W
from IPython.display import display, clear_output

# --- STAC / raster ---
from pystac_client import Client
import rasterio
from rasterio.enums import Resampling
from pyproj import Transformer

warnings.filterwarnings("ignore")
%matplotlib inline

# ====================== UTIL / ESTADO GLOBAL ======================
ESCALAS = ['25k','50k','100k','250k','1kk']

quadricula = {}         # dict: {fid: {'folha': Series(...EPSG...), 'gama_*': df, 'mag_*': df, 'geof_*': df, ...}}
data_grid = None        # nome da camada interpolada ativa (ex.: 'geof_1105_linear')
som_store = {}          # {k: {'som', 'imp', 'sca', 'feats', 'layer'}}
som_last_pred = None    # {'k','classes','metas','fids','feats','layer'}

bdc_items = []          # lista de pystac.Items da última busca

# ---------- helpers genéricos ----------
def _ids_from_mc(escala, filtro_regex=None):
    mc = import_malha_cartog(escala=escala)
    ids = mc['id_folha'].astype(str).tolist()
    if filtro_regex:
        pat = re.compile(filtro_regex, re.IGNORECASE)
        ids = [i for i in ids if pat.search(i)]
    return sorted(ids)

def _scan_layers_from_quadricula(q):
    layers = set()
    for fid, blob in (q or {}).items():
        for k, v in blob.items():
            if isinstance(v, pd.DataFrame):
                layers.add(k)
    return tuple(sorted(layers))

def _available_columns(q, layers):
    cols = set()
    for fid, blob in (q or {}).items():
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in df.columns:
                    if c in ('X','Y','E_utm','N_utm'):
                        continue
                    cols.add(c)
    cols = sorted(cols, key=lambda c: (c!='MDT', c))
    return tuple(cols)

def _global_min_max_numeric(q, ids, layers, column, remove_negatives=False):
    vals = []
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                s = df[column]
                if pd.api.types.is_numeric_dtype(s):
                    if remove_negatives:
                        s = s[s >= 0]
                    if s.size:
                        vals.append(s.to_numpy())
    if not vals:
        return None, None
    v = np.concatenate(vals)
    if v.size == 0 or np.all(np.isnan(v)):
        return None, None
    return float(np.nanmin(v)), float(np.nanmax(v))

def _plot_layers_for_column(q, ids, layers, column, remove_negatives=False):
    plt.figure(figsize=(12, 9))
    ax = plt.gca()
    is_numeric = False
    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame) and column in df.columns:
                is_numeric = pd.api.types.is_numeric_dtype(df[column])
                break
        if is_numeric:
            break

    if is_numeric:
        vmin, vmax = _global_min_max_numeric(q, ids, layers, column, remove_negatives)
        if vmin is not None and vmax is not None and vmin == vmax:
            eps = 1e-9
            vmin, vmax = vmin - eps, vmax + eps
        norm = colors.Normalize(vmin=vmin, vmax=vmax) if vmin is not None else None
        cmap = cm.get_cmap('terrain')

    for fid in ids:
        blob = q.get(fid, {})
        for lay in layers:
            df = blob.get(lay)
            if not isinstance(df, pd.DataFrame) or column not in df.columns:
                continue
            d = df
            if is_numeric and remove_negatives:
                d = d[d[column] >= 0]
                if d.empty:
                    continue
            if is_numeric:
                ax.scatter(d.X.values, d.Y.values, c=d[column].values,
                           s=0.1, cmap=cmap, norm=norm, marker='H')
            else:
                codes, _ = pd.factorize(d[column], sort=True)
                ax.scatter(d.X.values, d.Y.values, c=codes, s=0.1,
                           cmap='tab20', marker='H')

    ax.set_aspect('equal')
    ax.set_title(f'Pré-visualização • {column} • {len(ids)} folha(s) • camadas: {", ".join(layers)}')
    plt.axis('scaled')

    if is_numeric and norm is not None:
        cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
        vmin, vmax = norm.vmin, norm.vmax
        mid = (vmin + vmax) / 2.0
        cbar.set_ticks([vmin, mid, vmax])
        cbar.ax.set_yticklabels([f'{vmin:.3g}', f'{mid:.3g}', f'{vmax:.3g}'])
        cbar.set_label(f'{column} (min→máx)', rotation=90)
    plt.show()

def _make_discrete_cmap(n):
    base = plt.get_cmap('tab20')
    if hasattr(base, 'colors') and len(base.colors) >= n:
        return ListedColormap(base.colors[:n], name=f'tab20_{n}')
    return plt.get_cmap('nipy_spectral', n)

def _infer_suffix_from_names(*names):
    for nm in names:
        if not nm:
            continue
        m = re.search(r'(\d{4})', str(nm))
        if m:
            return m.group(1)
    return '0000'

def _norm_name(name: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(name).lower())

_SYNONYMS = {
    'GMT'      : {'gmt','magigrf','magr','igrf','mag','gmtigrf','magig rf','magigrf'},
    'MDT'      : {'mdt','alte','altura'},
    'CTCOR'    : {'ctcor','ctc','ct'},
    'eTh'      : {'eth','eth_ppm','thc','th_ppm','ethppm','th'},
    'eU'       : {'eu','uc','u','euppm','u_ppm'},
    'KPERC'    : {'kperc','kc','k','kpct','k_percent'},
    'UTHRAZAO' : {'uthrazao','uratio','u_th','u/th','u_th_ratio'},
    'UKRAZAO'  : {'ukrazao','u_k','u/k','u_k_ratio'},
    'THKRAZAO' : {'thkrazao','th_k','th/k','th_k_ratio'},
}

def _find_source_column(df: pd.DataFrame, canonical: str) -> str | None:
    want = _norm_name(canonical)
    for c in df.columns:
        if _norm_name(c) == want:
            return c
    syns = _SYNONYMS.get(canonical, set())
    cols_norm = { _norm_name(c): c for c in df.columns }
    for s in syns:
        if s in cols_norm:
            return cols_norm[s]
    return None

def _source_order_for_feature(canonical: str) -> list[str]:
    if canonical in ('GMT', 'MDT'):
        return ['mag', 'gama']
    return ['gama', 'mag']

# --------- INTERPOLAÇÃO para grade sintética ----------
def _interpolate_current_selection(quadricula, ids, gama_key, mag_key, features, psize, algo, noneg=False):
    suf = _infer_suffix_from_names(gama_key, mag_key)
    out_name = f"geof_{suf}_{algo}"

    for fid in ids:
        blob = quadricula.get(fid, {})
        gama_df = blob.get(gama_key)
        mag_df  = blob.get(mag_key)
        if (gama_df is None) and (mag_df is None):
            continue

        xu, yu = sintetic_grid(quadricula, fid, psize=int(psize))  # grid

        sources = {}
        if isinstance(gama_df, pd.DataFrame):
            gsrc = gama_df.copy()
            if noneg:
                for c in gsrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(gsrc[c]):
                        gsrc.loc[gsrc[c] < 0, c] = np.nan
            sources['gama'] = (np.asarray(gsrc['X']), np.asarray(gsrc['Y']), gsrc)
        if isinstance(mag_df, pd.DataFrame):
            msrc = mag_df.copy()
            if noneg:
                for c in msrc.columns:
                    if c not in ('X','Y') and pd.api.types.is_numeric_dtype(msrc[c]):
                        msrc.loc[msrc[c] < 0, c] = np.nan
            sources['mag'] = (np.asarray(msrc['X']), np.asarray(msrc['Y']), msrc)

        if not sources:
            continue

        out = {'X': xu, 'Y': yu}
        for f in features:
            arr = None
            for src in _source_order_for_feature(f):
                if src not in sources:
                    continue
                x, y, df = sources[src]
                real_col = _find_source_column(df, f)
                if real_col is None:
                    continue
                vals = df[real_col].to_numpy()
                arr = interp_at(x, y, vals, xu, yu, algorithm=algo, extrapolate=True)
                break
            if arr is None:
                arr = np.full_like(xu, np.nan, dtype='float32')
            out[f] = arr

        quadricula[fid][out_name] = pd.DataFrame(out)

    return out_name

# ---------------- helpers de UI (rescan) ----------------
def _rescan_from_quadricula():
    q = globals().get('quadricula', {})
    layers = _scan_layers_from_quadricula(q)
    w_layers.options = layers
    global data_grid
    pick = ()
    if data_grid and data_grid in layers:
        pick = (data_grid,)
    else:
        pref = [w_gama.value, w_mag.value]
        pick = tuple([p for p in pref if p in layers]) or (layers[:1] if layers else ())
    w_layers.value = pick

    cols = _available_columns(q, w_layers.value) or ('MDT',)
    w_cols.options = cols
    w_cols.value = tuple([c for c in ('MDT',) if c in cols]) or (cols[0],)

    numeric = []
    for fid, blob in q.items():
        for lay in w_layers.value:
            df = blob.get(lay)
            if isinstance(df, pd.DataFrame):
                for c in cols:
                    if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
                        numeric.append(c)
    opts = sorted(set(numeric), key=lambda c: (c!='MDT', c)) or ['MDT']
    w_feats.options = opts
    keep = [c for c in w_feats.value if c in opts] or (['MDT'] if 'MDT' in opts else opts[:min(5,len(opts))])
    w_feats.value = tuple(keep)

    test_opts = []
    if data_grid:
        for fid, blob in q.items():
            if data_grid in blob and isinstance(blob[data_grid], pd.DataFrame):
                test_opts.append(fid)
    w_test_ids.options = tuple(sorted(test_opts))
    w_test_ids.value = tuple(sorted(test_opts))[:min(4, len(test_opts))]

    ks = sorted(list(som_store.keys()))
    w_k_apply.options = ks
    if ks:
        w_k_apply.value = ks[0]

def _normalize_xy(df):
    if not {'E_utm','N_utm'}.issubset(df.columns):
        if {'X','Y'}.issubset(df.columns):
            df = df.rename(columns={'X':'E_utm','Y':'N_utm'}).copy()
        else:
            raise ValueError("Camada sem colunas de coordenadas ('X','Y' ou 'E_utm','N_utm').")
    df = df.sort_values(['N_utm','E_utm'], ascending=[False, True], ignore_index=True, kind='mergesort')
    xs1d = np.sort(df['E_utm'].unique())
    ys1d = np.sort(df['N_utm'].unique())
    nx, ny = xs1d.size, ys1d.size
    xs_mesh, ys_mesh = np.meshgrid(xs1d, ys1d)  # (ny, nx)
    return df, xs_mesh, ys_mesh, nx, ny

def _build_matrix_for_fids(quadricula, features, layer, fids=None):
    fids_all = sorted(quadricula.keys()) if fids is None else list(fids)
    all_blocks, slc, metas = [], {}, {}
    k = 0
    for fid in fids_all:
        blob = quadricula.get(fid, {})
        if layer not in blob:
            continue
        df = blob[layer].copy()
        try:
            df, xs_mesh, ys_mesh, nx, ny = _normalize_xy(df)
        except Exception:
            continue
        metas[fid] = {'nx': nx, 'ny': ny, 'xs': xs_mesh, 'ys': ys_mesh}
        X = df[features].to_numpy(dtype='float32')
        if X.size == 0:
            continue
        all_blocks.append(X)
        slc[fid] = slice(k, k + len(X)); k += len(X)
    if not all_blocks:
        raise RuntimeError(f"Nenhuma folha com '{layer}' e as features escolhidas foi encontrada.")
    return np.vstack(all_blocks), slc, metas

def _qe(som, X_std):
    D = som.transform(X_std)
    return float(np.mean(np.min(D, axis=1)))

def _te_1d(som, X_std):
    D = som.transform(X_std)
    bmu = np.argmin(D, axis=1)
    D2 = D.copy(); D2[np.arange(D.shape[0]), bmu] = np.inf
    sbmu = np.argmin(D2, axis=1)
    return float(np.mean(np.abs(bmu - sbmu) > 1))

def _make_discrete_cmap(n):
    base = plt.get_cmap('tab20')
    if hasattr(base, 'colors') and len(base.colors) >= n:
        return ListedColormap(base.colors[:n], name=f'tab20_{n}')
    return plt.get_cmap('nipy_spectral', n)

def _plot_classes(classes_by_fid, metas, n_clusters, flip_ns=False, titulo='Mapa preditivo (SOM)'):
    cmap = _make_discrete_cmap(n_clusters)
    bounds = np.arange(-0.5, n_clusters + 0.5, 1)
    norm = BoundaryNorm(bounds, ncolors=n_clusters, clip=True)
    fig, ax = plt.subplots(figsize=(10, 10), facecolor='w')
    for fid in sorted(classes_by_fid.keys()):
        Z = classes_by_fid[fid]
        if flip_ns:
            Z = np.flipud(Z)
        xs = metas[fid]['xs']; ys = metas[fid]['ys']
        ax.pcolormesh(xs, ys, Z, cmap=cmap, norm=norm, shading='nearest', rasterized=True)
    ax.set_aspect('equal'); ax.set_title(titulo)
    cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, ticks=np.arange(n_clusters), pad=0.01)
    cbar.ax.set_yticklabels([f'Classe {i+1}' for i in range(n_clusters)])
    cbar.set_label('Classes')
    plt.tight_layout(); plt.show()

def _predict_per_folha(som, X_std, slc, metas):
    out = {}
    for fid, s in slc.items():
        y = som.predict(X_std[s])
        ny, nx = metas[fid]['ny'], metas[fid]['nx']
        out[fid] = y.reshape(ny, nx)
    return out

# ------- tabela longa + boxplots --------
def som_build_long_table(quadricula, layer, classes_by_fid, metas, atributos, fids=None):
    rows = []
    fids_iter = list(classes_by_fid.keys()) if fids is None else list(fids)
    for fid in fids_iter:
        if fid not in classes_by_fid:
            continue
        Z = classes_by_fid[fid]
        blob = quadricula.get(fid, {})
        if layer not in blob:
            continue
        df = blob[layer].copy()
        df, xs_mesh, ys_mesh, nx, ny = _normalize_xy(df)
        if Z.shape != (ny, nx):
            Zv = np.ravel(Z)[:ny*nx]
        else:
            Zv = Z.ravel(order='C')
        cols_keep = []
        for a in atributos:
            if a in df.columns:
                cols_keep.append(a)
        sub = pd.DataFrame({
            'fid': fid,
            'E_utm': df['E_utm'].to_numpy(),
            'N_utm': df['N_utm'].to_numpy(),
            'classe': Zv.astype(int)
        })
        for a in cols_keep:
            sub[a] = df[a].to_numpy()
        rows.append(sub)
    if not rows:
        raise RuntimeError("Nenhum dado disponível para montar a tabela longa (verifique layer/atributos).")
    out = pd.concat(rows, axis=0, ignore_index=True)
    return out

def plot_boxplots_por_atributo(
    df_long: pd.DataFrame,
    atributos: list[str],
    classes: list | None = None,
    ncols: int = 2,
    showfliers: bool = False,
    rotation: int = 45,
    sharey: bool = True,
    figsize_cell: tuple[float, float] = (4.0, 3.2),
    suptitle: str | None = None,
):
    import math
    if classes is None:
        classes = sorted(pd.Series(df_long['classe']).dropna().unique())
    n_classes = len(classes)
    if n_classes == 0:
        print("[Boxplot] Nenhuma classe para plotar.")
        return None
    ncols = max(1, int(ncols)); nrows = math.ceil(n_classes / ncols)
    fig_w = max(6.0, figsize_cell[0] * ncols)
    fig_h = max(3.2, figsize_cell[1] * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), sharey=sharey)
    axes = np.atleast_1d(axes).ravel()

    global_vals = []
    if sharey:
        for c in classes:
            sub = df_long[df_long['classe'] == c]
            for a in atributos:
                if a in sub.columns:
                    s = pd.to_numeric(sub[a], errors='coerce').dropna().values
                    if s.size:
                        global_vals.append(s)
        if global_vals:
            gcat = np.concatenate(global_vals)
            y_min, y_max = np.nanmin(gcat), np.nanmax(gcat)
            if not np.isfinite(y_min) or not np.isfinite(y_max) or y_min == y_max:
                y_min, y_max = None, None
        else:
            y_min, y_max = None, None
    else:
        y_min = y_max = None

    for i, c in enumerate(classes):
        ax = axes[i]
        sub = df_long[df_long['classe'] == c]
        vals_list, labels = [], []
        for a in atributos:
            if a not in sub.columns:
                continue
            s = pd.to_numeric(sub[a], errors='coerce').dropna()
            if s.size:
                vals_list.append(s.values)
                labels.append(a)
        try:
            c_label = int(c) + 1
        except Exception:
            c_label = c
        if not vals_list:
            ax.set_title(f"Classe {c_label} (sem dados)"); ax.axis("off")
        else:
            ax.boxplot(vals_list, labels=labels, showfliers=showfliers)
            ax.set_title(f"Classe {c_label}")
            ax.set_xlabel("Atributo"); ax.set_ylabel("Valor")
            ax.tick_params(axis='x', labelrotation=rotation)
            if y_min is not None and y_max is not None:
                pad = 0.03 * (y_max - y_min if y_max != y_min else 1.0)
                ax.set_ylim(y_min - pad, y_max + pad)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()
    return fig

# ============================ WIDGETS BASE ============================
w_escala = W.Dropdown(options=ESCALAS, value='100k', description='Escala')
w_filtro = W.Text(placeholder='ex.: SF23_YB', description='Filtro')
w_ids    = W.SelectMultiple(options=(), rows=10, description='Folhas')
w_selall = W.ToggleButton(value=False, description='Selecionar tudo', icon='check')
w_clear  = W.Button(description='Limpar', icon='trash')

w_ext    = W.IntSlider(min=0, max=2000, step=100, value=600, description='extend_size')
w_gama   = W.Dropdown(options=['gama_line_1105','gama_line_1089','gama_1039','gama_3022'], value='gama_line_1105', description='Gama')
w_mag    = W.Dropdown(options=['mag_line_1105','mag_line_1089','mag_1039','mag_3022'], value='mag_line_1105', description='Mag')
w_load   = W.Button(description='Carregar brutos', button_style='success', icon='download')

w_feats_interp = W.SelectMultiple(
    options=['GMT','CTCOR','eTh','eU','KPERC','UTHRAZAO','UKRAZAO','THKRAZAO','MDT'],
    value=('GMT','CTCOR','eTh','eU','KPERC','MDT'),
    rows=8, description='Features (grid)'
)
w_psize  = W.IntSlider(min=50, max=1000, step=50, value=200, description='Pixel (m)')
w_algo   = W.Dropdown(options=[('Linear','linear'),('Cúbico','cubic')], value='linear', description='Algoritmo')
w_nonegI = W.Checkbox(value=False, description='Negativos→NaN (grid)')
w_interpolar = W.Button(description='Interpolar grade', icon='shuffle')

w_layers = W.SelectMultiple(options=(), rows=6, description='Camadas')
w_cols   = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=6, description='Colunas')
w_nonegP = W.Checkbox(value=False, description='Remover negativos (preview)')
w_refresh = W.Button(description='Atualizar', icon='refresh')
w_plot   = W.Button(description='Pré-visualizar', icon='eye')

w_feats  = W.SelectMultiple(options=('MDT',), value=('MDT',), rows=8, description='Features (SOM)')
w_sigma  = W.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.5, description='sigma')
w_iter   = W.IntSlider(min=500, max=30000, step=500, value=10000, description='max_iter')
w_seed   = W.IntSlider(min=0, max=9999, step=1, value=42, description='seed')
w_flip   = W.Checkbox(value=False, description='flip N-S no plot')

w_ks_train = W.SelectMultiple(options=tuple(range(3, 31)), value=(8, 12, 16), rows=8, description='k p/ treinar')
w_train  = W.Button(description='Treinar SOM(s)', button_style='primary', icon='play')

w_test_ids = W.SelectMultiple(options=(), rows=8, description='Folhas (teste)')
w_seltest  = W.ToggleButton(value=False, description='Selecionar todas (teste)', icon='check')
w_k_apply  = W.Dropdown(options=[], description='k (aplicar)')
w_apply    = W.Button(description='Aplicar/Testar', icon='check-circle')
w_evalall  = W.Button(description='Comparar Ks (métricas)', icon='bar-chart')
w_clear_models = W.Button(description='Limpar modelos', icon='trash')

w_boxplots = W.Button(description='Boxplots por atributo', icon='bar-chart', button_style='')
w_datagrid_label = W.HTML(value="<b>Camada SOM:</b> <i>—</i>")
w_models_label   = W.HTML(value="<b>Modelos treinados:</b> <i>—</i>")
w_out    = W.Output()

# ============================ CALLBACKS BASE ============================
def refresh_ids(*_):
    ids = _ids_from_mc(w_escala.value, w_filtro.value.strip() or None)
    w_ids.options = ids
    w_selall.value = False

def on_selall_change(ch):
    if ch['name'] == 'value':
        w_ids.value = tuple(w_ids.options) if ch['new'] else ()

def on_seltest_change(ch):
    if ch['name'] == 'value':
        w_test_ids.value = tuple(w_test_ids.options) if ch['new'] else ()

def on_clear_clicked(_):
    w_filtro.value = ''
    w_ids.value = ()

def on_load_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        print('# Montando grade…')
        quad = Build_mc(escala=w_escala.value, ID=list(w_ids.value), verbose=True)
        print('# Carregando dados brutos…')
        _g, _m = Upload_geof(
            quad,
            gama_xyz=w_gama.value,
            mag_xyz=w_mag.value,
            extend_size=int(w_ext.value)
        )
        quad = pop_nodata(quad)
        globals()['quadricula'] = quad
        print(f'Folhas ativas: {len(quad)}')
        globals()['data_grid'] = None
        w_datagrid_label.value = "<b>Camada SOM:</b> <i>— (interpole primeiro)</i>"
        som_store.clear(); globals()['som_last_pred'] = None
        w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        print('Pronto. Dados brutos anexados. Agora execute a INTERPOLAÇÃO.')

def on_refresh_clicked(_):
    with w_out:
        clear_output()
        if 'quadricula' not in globals():
            print('A variável global `quadricula` ainda não existe. Carregue dados primeiro.')
            return
        print('Re-escaneando `quadricula`…')
        _rescan_from_quadricula()
        print('Atualizado.')

def on_plot_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        if not w_layers.value:
            print('Nenhuma camada selecionada.')
            return
        q = globals().get('quadricula', {})
        for col in w_cols.value:
            _plot_layers_for_column(q, w_ids.value, w_layers.value, col, remove_negatives=w_nonegP.value)

def on_interpolar_clicked(_):
    with w_out:
        clear_output()
        if not w_ids.value:
            print('Selecione ao menos 1 folha.')
            return
        feats_grid = list(w_feats_interp.value)
        if not feats_grid:
            print('Selecione ao menos 1 feature para a grade.')
            return
        q = globals().get('quadricula', {})
        if not q:
            print('Carregue os dados brutos primeiro.')
            return
        print(f"# Interpolando (algo={w_algo.value}, pixel={int(w_psize.value)} m)…")
        out_layer = _interpolate_current_selection(
            q, ids=w_ids.value,
            gama_key=w_gama.value, mag_key=w_mag.value,
            features=feats_grid, psize=int(w_psize.value),
            algo=w_algo.value, noneg=w_nonegI.value
        )
        globals()['quadricula'] = q
        globals()['data_grid'] = out_layer
        w_datagrid_label.value = f"<b>Camada SOM:</b> <code>{out_layer}</code>"
        print(f"→ Camada criada: {out_layer}")
        som_store.clear(); globals()['som_last_pred'] = None
        w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
        _rescan_from_quadricula()
        if out_layer in w_layers.options:
            w_layers.value = (out_layer,)

def on_train_clicked(_):
    with w_out:
        clear_output()
        feats = list(w_feats.value)
        if not feats:
            print("Selecione ao menos 1 feature (SOM)."); return
        if not globals().get('data_grid'):
            print("Interpole a grade primeiro (botão 'Interpolar grade')."); return
        layer = globals()['data_grid']
        q = globals().get('quadricula', {})
        if not q:
            print('Carregue dados e interpele a grade antes do SOM.'); return

        print(f"[TREINO] Montando matriz global a partir de '{layer}'…")
        X_all, _, _ = _build_matrix_for_fids(q, feats, layer, fids=None)

        imp = SimpleImputer(strategy='median')
        X_imp = imp.fit_transform(X_all)
        sca   = StandardScaler().fit(X_imp)
        X_std = sca.transform(X_imp)

        ks = sorted(set(int(k) for k in w_ks_train.value))
        if not ks:
            print("Selecione ao menos um valor de k para treinar."); return

        np.random.seed(int(w_seed.value))
        trained = []
        for k in ks:
            print(f" - Treinando SOM(k={k}, sigma={float(w_sigma.value)}, it={int(w_iter.value)}) …")
            som = SOM(m=int(k), n=1, sigma=float(w_sigma.value), dim=len(feats), max_iter=int(w_iter.value))
            som.fit(X_std)
            som_store[k] = {'som': som, 'imp': imp, 'sca': sca, 'feats': feats, 'layer': layer}
            trained.append(k)

        if trained:
            w_models_label.value = f"<b>Modelos treinados:</b> {', '.join(map(str, sorted(som_store.keys())))}"
            w_k_apply.options = sorted(list(som_store.keys()))
            w_k_apply.value = w_k_apply.options[0]
            print("Modelos treinados com sucesso.")
        else:
            print("Nenhum modelo foi treinado.")

def on_apply_clicked(_):
    with w_out:
        clear_output()
        if not som_store:
            print("Nenhum modelo treinado. Use 'Treinar SOM(s)' antes."); return
        if not w_test_ids.value:
            print("Selecione ao menos 1 folha para teste/aplicação."); return
        k = int(w_k_apply.value)
        model = som_store.get(k)
        if model is None:
            print(f"k={k} não encontrado entre os modelos treinados."); return
        feats = model['feats']; layer = model['layer']; q = globals().get('quadricula', {})

        print(f"[TESTE] Preparando subset ({len(w_test_ids.value)} folha(s)) com layer '{layer}'…")
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e:
            print(str(e)); return
        X_te_std = model['sca'].transform(model['imp'].transform(X_te))

        qe = _qe(model['som'], X_te_std)
        te = _te_1d(model['som'], X_te_std)
        df_metrics = pd.DataFrame([{'k': k, 'QE_test': qe, 'TE_test': te}])
        print(df_metrics.to_string(index=False))

        classes = _predict_per_folha(model['som'], X_te_std, slc_te, metas_te)
        _plot_classes(
            classes, metas_te, n_clusters=k, flip_ns=bool(w_flip.value),
            titulo=f"SOM (aplicar): k={k} | sigma={float(w_sigma.value)} | it={int(w_iter.value)} | layer={layer}"
        )
        globals()['som_last_pred'] = {
            'k': k, 'classes': classes, 'metas': metas_te,
            'fids': tuple(w_test_ids.value), 'feats': tuple(feats), 'layer': layer
        }
        print("[SOM] Predição salva: som_last_pred (k, classes, metas, fids, feats, layer).")

def on_evalall_clicked(_):
    with w_out:
        clear_output()
        if not som_store:
            print("Nenhum modelo treinado. Use 'Treinar SOM(s)' antes."); return
        if not w_test_ids.value:
            print("Selecione ao menos 1 folha para avaliação."); return
        any_k = next(iter(som_store))
        feats = som_store[any_k]['feats']; layer = som_store[any_k]['layer']
        q = globals().get('quadricula', {})
        try:
            X_te, slc_te, metas_te = _build_matrix_for_fids(q, feats, layer, fids=w_test_ids.value)
        except RuntimeError as e:
            print(str(e)); return
        rows = []
        for k in sorted(som_store.keys()):
            model = som_store[k]
            if model['feats'] != feats or model['layer'] != layer:
                rows.append({'k': k, 'QE_test': np.nan, 'TE_test': np.nan, 'obs': 'incompatível (feats/layer)'})
                continue
            X_te_std = model['sca'].transform(model['imp'].transform(X_te))
            rows.append({'k': k, 'QE_test': _qe(model['som'], X_te_std), 'TE_test': _te_1d(model['som'], X_te_std)})
        df = pd.DataFrame(rows).sort_values('QE_test', ascending=True, na_position='last', ignore_index=True)
        print(df.to_string(index=False))

def on_clear_models_clicked(_):
    som_store.clear(); globals()['som_last_pred'] = None
    w_models_label.value = "<b>Modelos treinados:</b> <i>—</i>"
    w_k_apply.options = []
    with w_out:
        clear_output()
        print("Modelos apagados.")

def on_boxplots_clicked(_):
    with w_out:
        clear_output()
        if not som_store:
            print("Nenhum modelo treinado. Treine e aplique um SOM primeiro."); return
        lp = globals().get('som_last_pred')
        if lp is None:
            print("Nenhuma predição recente encontrada. Clique em 'Aplicar/Testar' e tente novamente."); return
        k = lp['k']; classes = lp['classes']; metas = lp['metas']
        fids = lp['fids']; feats = list(lp['feats']); layer = lp['layer']
        q = globals().get('quadricula', {})
        try:
            df_long = som_build_long_table(q, layer, classes, metas, atributos=feats, fids=fids)
        except RuntimeError as e:
            print(str(e)); return
        print(f"[Boxplots] {len(fids)} folha(s), k={k}, layer='{layer}', atributos={feats}")
        plot_boxplots_por_atributo(df_long, atributos=feats, ncols=2, showfliers=False)

# liga widgets base
w_escala.observe(refresh_ids, names='value')
w_filtro.observe(refresh_ids, names='value')
w_selall.observe(on_selall_change, names='value')
w_seltest.observe(on_seltest_change, names='value')
w_clear.on_click(on_clear_clicked)
w_load.on_click(on_load_clicked)
w_refresh.on_click(on_refresh_clicked)
w_plot.on_click(on_plot_clicked)
w_interpolar.on_click(on_interpolar_clicked)
w_train.on_click(on_train_clicked)
w_apply.on_click(on_apply_clicked)
w_evalall.on_click(on_evalall_clicked)
w_clear_models.on_click(on_clear_models_clicked)
w_boxplots.on_click(on_boxplots_clicked)

refresh_ids()

# ============================ BDC / STAC (INPE) ============================
BDC_ENDPOINT = "https://data.inpe.br/bdc/stac/v1"

def _aoi_bbox_from_ids(escala: str, ids: list[str]) -> list[float] | None:
    if not ids:
        return None
    gdf = import_malha_cartog(escala=escala)
    gdf = gdf[gdf['id_folha'].astype(str).isin([str(i) for i in ids])].copy()
    if gdf.empty:
        return None
    if gdf.crs is None:
        try:
            epsg = int(gdf.get('EPSG').dropna().iloc[0])
            gdf = gdf.set_crs(epsg)
        except Exception:
            pass
    try:
        gdf = gdf.to_crs(4326)
    except Exception:
        pass
    minx, miny, maxx, maxy = gdf.total_bounds
    return [float(minx), float(miny), float(maxx), float(maxy)]

def _bdc_list_collections(pattern: str | None = None) -> list[str]:
    cli = Client.open(BDC_ENDPOINT)
    cols = [c.id for c in cli.get_collections()]
    if pattern:
        pat = re.compile(pattern, re.IGNORECASE)
        cols = [c for c in cols if pat.search(c)]
    return sorted(cols)

def _bdc_search_items(collections, bbox, dt_range: str, cloud_min: int, cloud_max: int, limit: int, sort_dir: str):
    cli = Client.open(BDC_ENDPOINT)
    q = {"eo:cloud_cover": {"gte": int(cloud_min), "lte": int(cloud_max)}}
    sortby = ["properties.datetime"] if sort_dir == "asc" else ["-properties.datetime"]
    search = cli.search(collections=list(collections), bbox=bbox, datetime=dt_range, query=q, sortby=sortby, max_items=limit)
    return list(search.items())

def _bdc_pick_visual_asset(item) -> tuple[str, str] | None:
    for key in ("tci", "visual", "overview", "thumbnail"):
        a = item.assets.get(key)
        if a and a.href:
            return a.href, key
    for trip in (("red","green","blue"), ("B4","B3","B2"), ("B3","B2","B1")):
        if all(k in item.assets for k in trip):
            return item.assets[trip[0]].href, trip[0]
    return None

def _bdc_preview_thumbs(items, max_show=12):
    n = min(len(items), max_show)
    if n == 0:
        print("Nenhum item."); return
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 2.8*nrows))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        it = items[i]
        ax = axes[i]
        pair = _bdc_pick_visual_asset(it)
        ax.axis("off")
        title = f"{it.collection_id}\n{getattr(it, 'datetime', None).date() if getattr(it,'datetime',None) else '—'}"
        ax.set_title(title, fontsize=9)
        if pair is None:
            ax.text(0.5, 0.5, "sem preview", ha="center", va="center"); continue
        href, key = pair
        try:
            r = requests.get(href, timeout=15); r.raise_for_status()
            import PIL.Image as Image
            from io import BytesIO
            img = Image.open(BytesIO(r.content))
            ax.imshow(img)
        except Exception as e:
            ax.text(0.5, 0.5, f"erro preview\n{e}", ha="center", va="center", fontsize=8)
    for j in range(i+1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout(); plt.show()

# --------- Amostragem de bandas na grade SOM ----------
def _grid_epsg_from_blob(blob):
    v = blob.get('folha', None)
    if v is not None:
        try:
            return int(getattr(v, 'EPSG', v.get('EPSG')))
        except Exception:
            try:
                return int(v['EPSG'])
            except Exception:
                pass
    if 'EPSG' in blob:
        try:
            return int(blob['EPSG'])
        except Exception:
            pass
    raise RuntimeError("Não foi possível inferir o EPSG da folha (verifique quadricula[fid]['folha']['EPSG']).")

def _open_remote_raster(href):
    try:
        return rasterio.open(href)
    except Exception:
        pass
    if not href.startswith('/vsicurl/'):
        return rasterio.open('/vsicurl/' + href)
    raise

def _resolve_band_assets(item, bands_text):
    """
    Retorna [(name, href, idxs)] a partir de 'bands_text'.
    Suporta: 'tci'/'visual', 'B4,B3,B2' (ou B04,B03,B02), 'red,green,blue' etc.
    """
    wanted = [b.strip() for b in bands_text.split(',') if b.strip()]
    out = []
    assets_ci = {k.lower(): k for k in item.assets.keys()}

    def _pick(*keys):
        for k in keys:
            kk = assets_ci.get(k.lower())
            if kk and getattr(item.assets[kk], "href", None):
                return kk, item.assets[kk].href
        return None, None

    for w in wanted:
        lw = w.lower()
        if lw == 'tci':
            k, href = _pick('tci', 'visual')
            if not href:
                if (assets_ci.get('b4') and assets_ci.get('b3') and assets_ci.get('b2')) \
                   or (assets_ci.get('b04') and assets_ci.get('b03') and assets_ci.get('b02')):
                    raise RuntimeError("Item não tem 'tci'/'visual'. Selecione B4,B3,B2 (ou B04,B03,B02).")
                raise RuntimeError("Item não oferece 'tci'/'visual'.")
            out.append((w, href, (1, 2, 3))); continue

        kk = assets_ci.get(lw)
        if kk:
            out.append((kk, item.assets[kk].href, (1,))); continue

        if lw in ('red', 'b4', 'b04', 'band4'):
            k, href = _pick('B4', 'B04', 'red')
            if href: out.append(('red', href, (1,))); continue
        if lw in ('green', 'b3', 'b03', 'band3'):
            k, href = _pick('B3', 'B03', 'green')
            if href: out.append(('green', href, (1,))); continue
        if lw in ('blue', 'b2', 'b02', 'band2'):
            k, href = _pick('B2', 'B02', 'blue')
            if href: out.append(('blue', href, (1,))); continue

        m = re.fullmatch(r'b(?:and)?0?(\d+)', lw)
        if m:
            n = int(m.group(1))
            k, href = _pick(f'B{n}', f'B{n:02d}')
            if href:
                out.append((f'B{n}', href, (1,))); continue

        raise RuntimeError(f"Banda/asset '{w}' não encontrada no item.")
    return out

def _sample_asset_into_layer(quadricula, fids, layer_name, href, band_idxs=(1,), prefix='sat', resampling='bilinear'):
    with _open_remote_raster(href) as ds:
        if ds.crs is None:
            raise RuntimeError("GeoTIFF sem CRS. Não é possível projetar.")
        resamp = Resampling.bilinear if str(resampling).startswith('bil') else Resampling.nearest
        ok_cols = 0
        for fid in fids:
            blob = quadricula.get(fid, {})
            if layer_name not in blob or not isinstance(blob[layer_name], pd.DataFrame):
                continue
            df = blob[layer_name]
            if not {'X','Y'}.issubset(df.columns):
                continue
            try:
                epsg_grid = _grid_epsg_from_blob(blob)
            except Exception as e:
                print(f" - {fid}: erro EPSG → {e}")
                continue
            tr = Transformer.from_crs(f"EPSG:{epsg_grid}", ds.crs, always_xy=True)
            xx, yy = tr.transform(df['X'].to_numpy(), df['Y'].to_numpy())

            def _batched(xa, ya, bs=200000):
                for i in range(0, xa.size, bs):
                    yield xa[i:i+bs], ya[i:i+bs]

            for j, b in enumerate(band_idxs, 1):
                vals = np.full(df.shape[0], np.nan, dtype='float32')
                k = 0
                try:
                    for xb, yb in _batched(xx, yy):
                        pts = list(zip(xb, yb))
                        it = ds.sample(pts, indexes=b, resampling=resamp)
                        out = np.fromiter((row[0] for row in it), dtype='float32', count=xb.size)
                        vals[k:k+xb.size] = out
                        k += xb.size
                except Exception as e:
                    print(f" - {fid}: erro amostrando banda {b} → {e}")
                    continue

                if len(band_idxs) == 3:
                    suffix = ('r','g','b')[j-1] if j <= 3 else f'b{j}'
                    col = f"{prefix}_{suffix}"
                elif len(band_idxs) == 1:
                    col = f"{prefix}"
                else:
                    col = f"{prefix}_b{b}"

                df[col] = vals.astype('float32', copy=False)
                ok_cols += 1
        return ok_cols

def _format_item_label(it, i):
    coll = getattr(it, "collection_id", "") or ""
    dt   = getattr(it, "datetime", None)
    dts  = (dt.date().isoformat() if hasattr(dt, "date") else str(dt)) if dt else "—"
    props = getattr(it, "properties", {}) or {}
    cc = props.get("eo:cloud_cover") or props.get("cloud_cover")
    cc_str = (f"{cc:.0f}%" if isinstance(cc, (int, float)) else "—")
    return f"{i:02d} | {coll} | {dts} | clouds {cc_str}"

# --- Widgets BDC ---
w_bdc_filter = W.Text(placeholder='regex (ex.: landsat|sentinel|cbers)', description='Filtro')
w_bdc_list   = W.Button(description='Listar coleções', icon='list')
w_bdc_cols   = W.SelectMultiple(options=(), rows=8, description='Coleções')

w_bdc_date   = W.Text(value='2018-01-01/2025-12-31', description='Data (UTC)')
w_bdc_cloud  = W.IntRangeSlider(value=[0, 100], min=0, max=100, step=1, description='Nuvens (%)')
w_bdc_limit  = W.IntSlider(value=20, min=1, max=200, step=1, description='Limite')
w_bdc_sort   = W.Dropdown(options=[('Mais antigo','asc'), ('Mais recente','desc')], value='desc', description='Ordenar')

w_bdc_search = W.Button(description='Buscar itens', icon='search', button_style='info')
w_bdc_prev   = W.Button(description='Thumbnails', icon='image')
w_bdc_save   = W.Button(description='Baixar VISUAL', icon='download', button_style='success')

# novo: selecionar item e amostrar
w_bdc_item    = W.Dropdown(options=(), description='Item', disabled=True)
w_bdc_bands   = W.Text(value='tci', description='Bandas')        # ex.: 'tci' ou 'B4,B3,B2' ou 'red,green,blue' ou 'B8'
w_bdc_prefix  = W.Text(value='sat', description='Prefixo')       # prefixo das novas colunas
w_bdc_sample  = W.Button(description='Amostrar p/ grade', icon='plus-square', button_style='warning')
w_bdc_out     = W.Output()

def on_bdc_list_clicked(_):
    with w_bdc_out:
        clear_output()
        try:
            cols = _bdc_list_collections(w_bdc_filter.value.strip() or None)
            if not cols:
                print("Nenhuma coleção encontrada para o filtro.")
            else:
                w_bdc_cols.options = tuple(cols)
                print(f"{len(cols)} coleção(ões) listada(s). Selecione uma ou mais e pesquise.")
        except Exception as e:
            print("Erro ao listar coleções do BDC:", e)

def on_bdc_search_clicked(_):
    with w_bdc_out:
        clear_output()
        if not w_bdc_cols.value:
            print("Selecione ao menos 1 coleção.")
            return
        bbox = _aoi_bbox_from_ids(w_escala.value, list(w_ids.value))
        if not bbox:
            print("Selecione folhas (à esquerda) para definirmos a área.")
            return
        print("AOI (bbox WGS84):", bbox)
        try:
            items = _bdc_search_items(
                collections=w_bdc_cols.value,
                bbox=bbox,
                dt_range=w_bdc_date.value.strip(),
                cloud_min=w_bdc_cloud.value[0],
                cloud_max=w_bdc_cloud.value[1],
                limit=int(w_bdc_limit.value),
                sort_dir=w_bdc_sort.value
            )
        except Exception as e:
            print("Erro na busca STAC:", e)
            return

        globals()['bdc_items'] = items
        print(f"Encontrados {len(items)} item(ns). Use 'Thumbnails' ou selecione um Item para amostrar na grade.")
        labels = [_format_item_label(it, i) for i, it in enumerate(items)]
        w_bdc_item.options = list(zip(labels, range(len(items))))
        w_bdc_item.disabled = (len(items) == 0)
        if items:
            w_bdc_item.value = 0

def on_bdc_prev_clicked(_):
    with w_bdc_out:
        clear_output()
        _bdc_preview_thumbs(globals().get('bdc_items', []), max_show=16)

def on_bdc_save_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items:
            print("Nenhuma lista de itens disponível. Faça a busca primeiro.")
            return
        # baixa apenas asset 'visual/tci/overview/thumbnail' (rápido)
        os.makedirs("satellite_bdc", exist_ok=True)
        saved = []
        for it in items:
            pair = _bdc_pick_visual_asset(it)
            if not pair:
                continue
            href, key = pair
            name = os.path.basename(href.split('?')[0])
            fpath = os.path.join("satellite_bdc", f"{it.collection_id}_{it.id}_{key}_{name}")
            try:
                if not os.path.exists(fpath):
                    with requests.get(href, stream=True, timeout=60) as r:
                        r.raise_for_status()
                        with open(fpath, "wb") as f:
                            for ch in r.iter_content(1<<20):
                                if ch: f.write(ch)
                saved.append(fpath)
            except Exception as e:
                print(f"[WARN] Falha ao baixar {href}: {e}")
        if saved:
            print("Arquivos salvos:")
            for p in saved: print(" -", p)
        else:
            print("Nenhum asset visual pôde ser baixado.")

def on_bdc_sample_clicked(_):
    with w_bdc_out:
        clear_output()
        items = globals().get('bdc_items', [])
        if not items:
            print("Faça a busca primeiro (BDC → Buscar itens).")
            return
        idx = int(w_bdc_item.value)
        if not (0 <= idx < len(items)):
            print(f"Índice inválido. Escolha 0..{len(items)-1}.")
            return
        if not globals().get('data_grid'):
            print("Interpole a grade primeiro (crie a camada SOM).")
            return
        layer = globals()['data_grid']
        q = globals().get('quadricula', {})
        fids_target = [fid for fid, blob in q.items() if layer in blob]
        item = items[idx]
        try:
            bands = _resolve_band_assets(item, w_bdc_bands.value)
        except Exception as e:
            print("Bandas:", str(e)); return

        total_cols = 0
        print(f"Amostrando { [b[0] for b in bands] } → layer '{layer}' em {len(fids_target)} folha(s)…")
        for name, href, idxs in bands:
            try:
                cols = _sample_asset_into_layer(
                    q, fids_target, layer_name=layer, href=href,
                    band_idxs=tuple(idxs), prefix=(w_bdc_prefix.value or name),
                    resampling='bilinear'
                )
                total_cols += cols
                print(f"  - OK {name}: {cols} coluna(s) adicionada(s).")
            except Exception as e:
                print(f"  - {name}: erro → {e}")

        if total_cols == 0:
            print("Nenhuma coluna foi criada (verifique EPSG/GeoTIFF).")
        else:
            globals()['quadricula'] = q
            _rescan_from_quadricula()
            print("Pronto. As novas colunas já podem ser usadas no SOM.")

# liga widgets BDC
w_bdc_list.on_click(on_bdc_list_clicked)
w_bdc_search.on_click(on_bdc_search_clicked)
w_bdc_prev.on_click(on_bdc_prev_clicked)
w_bdc_save.on_click(on_bdc_save_clicked)
w_bdc_sample.on_click(on_bdc_sample_clicked)

# painel BDC
bdc_controls = W.VBox([
    W.HBox([w_bdc_filter, w_bdc_list]),
    W.HBox([w_bdc_cols]),
    W.HBox([w_bdc_date, w_bdc_cloud, w_bdc_limit, w_bdc_sort]),
    W.HBox([w_bdc_search, w_bdc_prev, w_bdc_save]),
    W.HBox([w_bdc_item, w_bdc_bands, w_bdc_prefix, w_bdc_sample]),
    w_bdc_out
])

# ============================ LAYOUT FINAL ============================
left = W.VBox([
    W.HBox([w_escala, w_filtro]),
    W.HBox([
        w_ids,
        W.VBox([w_selall, w_clear, w_ext, w_gama, w_mag, w_load, w_refresh, w_plot]),
    ]),
    W.HTML("<hr><b>Interpolação para grade</b>"),
    W.HBox([w_feats_interp, W.VBox([w_psize, w_algo, w_nonegI, w_interpolar])]),
    w_datagrid_label,
    W.HTML("<hr><b>Imagens de Satélite — BDC/INPE (STAC)</b>"),
    bdc_controls,
])

mid = W.VBox([
    W.HTML("<b>Pré-visualização</b>"),
    W.HBox([w_layers, w_cols]),
    w_nonegP,
    W.HTML("<hr><b>SOM — Treino</b>"),
    w_feats,
    W.HBox([w_sigma, w_iter, w_seed]),
    W.HBox([w_ks_train, w_train]),
    w_models_label
])

right = W.VBox([
    W.HTML("<b>SOM — Teste/Aplicação</b>"),
    W.HBox([w_test_ids, W.VBox([w_seltest, w_k_apply, w_apply, w_evalall, w_flip, w_clear_models, w_boxplots])])
])

ui = W.VBox([W.HBox([left, mid, right]), w_out])
display(ui)
